In [ ]:
# reorganize_and_explore.py
from pathlib import Path
import pandas as pd
import numpy as np
import keras
import matplotlib.pyplot as plt

# Use relative path from notebook location
# This works whether you're in notebooks/, scripts/, or project root
notebook_dir = Path.cwd()

# Find project root (where pyproject.toml is)
project_root = notebook_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:  # Reached filesystem root
        raise FileNotFoundError("Could not find project root (pyproject.toml not found)")

# Now use relative paths from project root
base_dir = project_root / "fall_detection_data"
processed_dir = base_dir / "processed"
models_dir = base_dir / "models"
models_dir.mkdir(exist_ok=True)
output_dir = models_dir

print(f"📂 Project root: {project_root}")
print(f"📂 Data directory: {base_dir}")
print(f"📂 Models directory: {models_dir}")
print()

print("=" * 80)
print("CURRENT DIRECTORY STRUCTURE")
print("=" * 80)

# Show current structure
for item in sorted(base_dir.iterdir()):
    if item.is_dir():
        print(f"\n📁 {item.name}/")
        # Show what's inside each directory
        sub_items = list(item.iterdir())[:5]
        for sub in sub_items:
            if sub.is_dir():
                file_count = len(list(sub.glob("*")))
                print(f"   📁 {sub.name}/ ({file_count} files)")
            else:
                print(f"   📄 {sub.name}")
        if len(list(item.iterdir())) > 5:
            print(f"   ... and {len(list(item.iterdir())) - 5} more")

print("\n" + "=" * 80)
print("PROPOSED REORGANIZATION")
print("=" * 80)

proposed_structure = """
fall_detection_data/
├── KFall/
│   ├── sensor_data/
│   │   ├── SA06/
│   │   │   ├── S06T01R01.csv  (KFall format: S##T##R##.csv)
│   │   │   ├── S06T02R01.csv
│   │   │   └── ...
│   │   └── SA07/ ...
│   └── labels/
│       ├── SA06_label.xlsx
│       └── SA07_label.xlsx ...
│
├── SisFall/
│   ├── SA01/
│   │   ├── D01_SA01_R01.txt  (SisFall format: <CODE>_<SUBJECT>_<TRIAL>.txt)
│   │   ├── F01_SA01_R01.txt
│   │   └── ...
│   ├── SA02/ ...
│   └── SE01/ ... (elderly subjects)
│
└── processed/
    ├── kfall_features.pkl
    ├── sisfall_features.pkl
    └── fused_dataset.pkl
"""

print(proposed_structure)

print("\n" + "=" * 80)
print("DATASET COMPARISON")
print("=" * 80)

# KFall structure
kfall_sensor = base_dir / "KFall" / "sensor_data"
if kfall_sensor.exists():
    kfall_subjects = sorted([d.name for d in kfall_sensor.iterdir() if d.is_dir()])
    sample_kfall = kfall_sensor / kfall_subjects[0]
    sample_kfall_file = list(sample_kfall.glob("*.csv"))[0]
    
    df_kfall = pd.read_csv(sample_kfall_file)
    
    print("\n📊 KFALL DATASET:")
    print(f"   Subjects: {len(kfall_subjects)} (SA06-SA38)")
    print(f"   Sampling Rate: 100 Hz (needs upsampling to 200 Hz)")
    print(f"   File Format: S##T##R##.csv")
    print(f"   Columns: {df_kfall.columns.tolist()}")
    print(f"   Data Shape (sample): {df_kfall.shape}")
    print(f"   Has Labels: ✅ Yes (temporal annotations in Excel files)")

# SisFall structure
sisfall_dir = base_dir / "SisFall"
if sisfall_dir.exists():
    sisfall_subjects = sorted([d.name for d in sisfall_dir.iterdir() if d.is_dir()])
    adults = [s for s in sisfall_subjects if s.startswith('SA')]
    elderly = [s for s in sisfall_subjects if s.startswith('SE')]
    
    sample_sisfall = sisfall_dir / adults[0]
    sample_sisfall_file = list(sample_sisfall.glob("*.txt"))[0]
    
    # Read SisFall file - more robust parsing
    try:
        # Method 1: Read line by line and parse manually
        with open(sample_sisfall_file, 'r') as f:
            lines = f.readlines()
        
        data = []
        for line in lines:
            # Remove semicolon and split by comma or whitespace
            line = line.strip().replace(';', '')
            values = line.replace(',', ' ').split()
            if len(values) == 9:  # Should have 9 columns
                data.append([float(v) for v in values])
        
        df_sisfall = pd.DataFrame(data)
        
        print("\n📊 SISFALL DATASET:")
        print(f"   Subjects: {len(sisfall_subjects)} total")
        print(f"     - Adults (SA): {len(adults)} (SA01-SA23)")
        print(f"     - Elderly (SE): {len(elderly)} (SE01-SE15)")
        print(f"   Sampling Rate: 200 Hz ✅")
        print(f"   File Format: <CODE>_<SUBJECT>_<TRIAL>.txt")
        print(f"   Columns: 9 (ADXL345: 0-2, ITG3200: 3-5, MMA8451Q: 6-8)")
        print(f"   Data Shape (sample): {df_sisfall.shape}")
        print(f"   Has Labels: ❌ No (must use Algorithm 1)")
        print(f"   Data Format: Raw bits (needs conversion to physical units)")
        
    except Exception as e:
        print(f"\n❌ Error reading SisFall file: {e}")
        print("   Will handle this in the preprocessing pipeline")

print("\n" + "=" * 80)
print("ACTIVITIES NEEDED FOR PAPER REPRODUCTION")
print("=" * 80)

print("\n📋 FROM KFALL (Table I):")
kfall_needed = {
    'T10': 'Stumble while walking',
    'T28': 'Vertical fall while walking (fainting)',
    'T30': 'Forward fall while walking (trip)',
    'T31': 'Forward fall while jogging (trip)',
    'T32': 'Forward fall while walking (slip)',
    'T33': 'Lateral fall while walking (slip)',
    'T34': 'Backward fall while walking (slip)'
}
for code, desc in kfall_needed.items():
    print(f"   {code}: {desc}")

print("\n📋 FROM SISFALL (Table I):")
print("\n   ADL Activities:")
sisfall_adl = {
    'D01': 'Walking slowly',
    'D02': 'Walking quickly',
    'D03': 'Jogging slowly',
    'D04': 'Jogging quickly',
    'D05': 'Walking upstairs/downstairs slowly',
    'D06': 'Walking upstairs/downstairs quickly',
    'D18': 'Stumble while walking'
}
for code, desc in sisfall_adl.items():
    print(f"   {code}: {desc}")

print("\n   Fall Activities:")
sisfall_falls = {
    'F01': 'Fall forward while walking (slip)',
    'F02': 'Fall backward while walking (slip)',
    'F03': 'Lateral fall while walking (slip)',
    'F04': 'Fall forward while walking (trip)',
    'F05': 'Fall forward while jogging (trip)',
    'F06': 'Vertical fall while walking (fainting)'
}
for code, desc in sisfall_falls.items():
    print(f"   {code}: {desc}")

print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("""
1. ✅ Data is properly organized
2. ⏭️  Implement preprocessing pipeline:
   - Load and convert SisFall raw bits to physical units
   - Upsample KFall from 100Hz to 200Hz
   - Apply Algorithm 1 for temporal segmentation
   - Extract features according to Table I
3. ⏭️  Z-score normalization and dataset fusion
4. ⏭️  Build and train FallNet""")


In [ ]:
# Load the newly processed data
X_data = np.load(processed_dir / "X_data.npy")
y_labels = np.load(processed_dir / "y_labels.npy")

# ============================================================================
# STEP 1: Merge Impact and Aftermath
# ============================================================================
print("Merging Impact and Aftermath classes...")
y_labels[y_labels == 7] = 6  # Change Aftermath (7) to Impact (6)

# ============================================================================
# STEP 2: Remove Fall_Recovery (NEW!)
# ============================================================================
print("\n" + "="*80)
print("REMOVING FALL_RECOVERY CLASS")
print("="*80)

from collections import Counter

# Show before
counts_before = Counter(y_labels)
print(f"\nBefore removal:")
print(f"  Total samples: {len(y_labels):,}")
print(f"  Fall_Recovery (class 4): {counts_before[4]} samples")

# Remove Fall_Recovery (class 4)
mask = y_labels != 4
X_data = X_data[mask]
y_labels_temp = y_labels[mask]

removed_count = (~mask).sum()
print(f"\n✅ Removed {removed_count} Fall_Recovery samples")

# Shift labels down (5→4, 6→5)
y_labels = y_labels_temp.copy()
y_labels[y_labels_temp > 4] -= 1  # Classes 5,6 become 4,5

print(f"\nAfter removal:")
print(f"  Total samples: {len(y_labels):,}")
print(f"  Removed: {removed_count} samples ({removed_count/(len(y_labels)+removed_count)*100:.2f}%)")

# ============================================================================
# STEP 3: Update label map (NOW 6 CLASSES: 0-5)
# ============================================================================
label_map = {
    'Walking': 0,
    'Jogging': 1,
    'Walking_stairs_updown': 2,
    'Stumble_while_walking': 3,
    'Fall_Initiation': 4,      # Was 5, now 4 ← SHIFTED DOWN!
    'Impact_Aftermath': 5,     # Was 6, now 5 ← SHIFTED DOWN!
}
reverse_label_map = {v: k for k, v in label_map.items()}

print(f"\n✅ Updated to 6 classes (0-5):")
for name, idx in sorted(label_map.items(), key=lambda x: x[1]):
    print(f"  Class {idx}: {name}")

y_categorical = keras.utils.to_categorical(y_labels, num_classes=6)  # ← HERE!
print(f"y_categorical shape: {y_categorical.shape}")

# ============================================================================
# DIAGNOSTICS
# ============================================================================
print("\n" + "="*80)
print("POST-REMOVAL DATA DIAGNOSTICS")
print("="*80)

# 1. Class distribution
class_counts = Counter(y_labels)
print("\n1. Class Distribution (6 classes):")
for cls_idx in sorted(class_counts.keys()):
    count = class_counts[cls_idx]
    pct = count / len(y_labels) * 100
    print(f"   Class {cls_idx} ({reverse_label_map[cls_idx]:30s}): {count:5d} ({pct:5.2f}%)")

# Calculate imbalance
max_count = max(class_counts.values())
min_count = min(class_counts.values())
print(f"\nImbalance ratio: {max_count/min_count:.2f}x (was 36.8x with Fall_Recovery)")

# 2. Per-class signal statistics
print("\n2. Per-Class Signal Statistics (Acc-Y axis):")
print(f"   {'Class':<35s} {'Mean':<10s} {'Std':<10s} {'Min':<10s} {'Max':<10s}")
print(f"   {'-'*75}")
for cls_idx in sorted(class_counts.keys()):
    class_samples = X_data[y_labels == cls_idx]
    acc_y = class_samples[:, :, 1]  # Y-axis acceleration
    
    mean_val = acc_y.mean()
    std_val = acc_y.std()
    min_val = acc_y.min()
    max_val = acc_y.max()
    
    print(f"   {reverse_label_map[cls_idx]:<35s} {mean_val:>8.4f}  {std_val:>8.4f}  {min_val:>8.2f}  {max_val:>8.2f}")

# 3. Variance ranking
print("\n3. Variance Ranking (Fall_Initiation should be #1):")
variances = []
for cls_idx in sorted(class_counts.keys()):
    class_samples = X_data[y_labels == cls_idx]
    acc_y_var = class_samples[:, :, 1].var()
    variances.append((reverse_label_map[cls_idx], acc_y_var, cls_idx))
variances.sort(key=lambda x: x[1], reverse=True)
for i, (name, var, idx) in enumerate(variances, 1):
    print(f"   {i}. {name:<35s}: {var:.4f}")

# 4. Visualize samples (update to 6 classes)
fig, axes = plt.subplots(3, 2, figsize=(15, 10))
axes = axes.flatten()
critical_classes = [
    label_map['Walking'],
    label_map['Fall_Initiation'],
    label_map['Impact_Aftermath'],
    label_map['Stumble_while_walking'],
    label_map['Jogging'],
    label_map['Walking_stairs_updown']
]
for i, cls_idx in enumerate(critical_classes):
    if cls_idx in class_counts:
        sample_idx = np.where(y_labels == cls_idx)[0][0]
        sample_data = X_data[sample_idx]
        
        time = np.arange(200) / 200
        axes[i].plot(time, sample_data[:, 0], label='Acc-X', alpha=0.7, linewidth=1)
        axes[i].plot(time, sample_data[:, 1], label='Acc-Y', alpha=0.7, linewidth=1)
        axes[i].plot(time, sample_data[:, 2], label='Acc-Z', alpha=0.7, linewidth=1)
        
        axes[i].set_title(f'{reverse_label_map[cls_idx]}', fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Time (s)')
        axes[i].set_ylabel('Normalized Acc')
        axes[i].legend(fontsize=8)
        axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("✅ DATA READY FOR TRAINING (6 CLASSES)")
print("="*80)

In [ ]:
# %% [markdown]
# # Save Preprocessed 6-Class Data
# Save the cleaned dataset after merging Impact/Aftermath and removing Fall_Recovery

# %%
import numpy as np
from pathlib import Path

# Setup paths (same as before)
base_dir = Path("~/repos/summerschool2023/projects/fall-detection/fall_detection_data").expanduser()
processed_dir = base_dir / "processed"

print("="*80)
print("SAVING PREPROCESSED 6-CLASS DATA")
print("="*80)

# Save the processed data
save_path_X = processed_dir / "X_data_6class.npy"
save_path_y = processed_dir / "y_labels_6class.npy"
save_path_y_cat = processed_dir / "y_categorical_6class.npy"

np.save(save_path_X, X_data)
np.save(save_path_y, y_labels)
np.save(save_path_y_cat, y_categorical)

print(f"\n✅ Saved preprocessed data:")
print(f"   X_data:        {save_path_X}")
print(f"   y_labels:      {save_path_y}")
print(f"   y_categorical: {save_path_y_cat}")

print(f"\nSaved shapes:")
print(f"   X_data:        {X_data.shape}")
print(f"   y_labels:      {y_labels.shape}")
print(f"   y_categorical: {y_categorical.shape}")

# Also save the label mapping for future reference
label_map_path = processed_dir / "label_map_6class.npy"
np.save(label_map_path, label_map)
     # ✅ Labels (numbers 0-5)

# Save the LABEL MAPPING (dictionary)
import json
with open(processed_dir / "label_map_6class.json", 'w') as f:
    json.dump(label_map, f, indent=2)                          # ✅ Class names → numbers
print(f"\n   label_map:     {label_map_path}")

print("\n" + "="*80)
print("✅ ALL DATA SAVED SUCCESSFULLY")
print("="*80)
print("\nTo load this data in future notebooks:")
print("```python")
print("X_data = np.load(processed_dir / 'X_data_6class.npy')")
print("y_labels = np.load(processed_dir / 'y_labels_6class.npy')")
print("y_categorical = np.load(processed_dir / 'y_categorical_6class.npy')")
print("label_map = np.load(processed_dir / 'label_map_6class.npy', allow_pickle=True).item()")
print("```")

In [ ]:
# %% [markdown]
# # FallNet CNN→LMU Hybrid Training Pipeline
# Sequential architecture: CNN spatial feature extraction → LMU temporal modeling
# Designed for fall detection comparison study (CNN-only vs CNN→LMU vs LMU-only)
# Target deployment: Arduino Nano 33 BLE Sense Rev2

# %%
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
from keras_lmu import LMU
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# %% [markdown]
## 1. FallNet Model Architectures

# %%
class FallNet:
    """
    FallNet: CNN-LMU Architectures for Pre-Impact Fall Detection
    
    Supports 4 model variants for ablation study:
      1. CNN-only         — spatial features only (embedded baseline)
      2. LMU-only         — temporal modeling on raw input
      3. CNN→LMU Hybrid   — CNN spatial extraction feeding LMU temporal backbone
      4. CNN+LMU Ensemble — parallel branches with averaged outputs (original FallNet)
    """
    
    def __init__(self, input_shape=(200, 6), n_classes=6):
        """
        Args:
            input_shape: (timesteps, features) = (200, 6) for 1s @ 200Hz, 6-axis IMU
            n_classes: Number of output classes (6 for our reduced set)
        """
        self.input_shape = input_shape
        self.n_classes = n_classes
        self.model = None
    
    # =========================================================================
    # Building Blocks
    # =========================================================================
    
    def _cnn_feature_extractor(self, inputs, name_prefix='cnn'):
        """
        CNN spatial feature extractor block.
        
        Takes raw (200, 6) IMU input and produces (50, 64) feature maps.
        Two conv+pool stages halve the sequence length twice: 200 → 100 → 50
        This gives the downstream LMU a shorter, richer sequence to process.
        
        Returns: feature tensor of shape (batch, 50, 64)
        """
        # First conv block: learn local IMU patterns (spike shapes, axis correlations)
        x = layers.Conv1D(
            filters=32,
            kernel_size=5,
            activation='relu',
            padding='same',
            kernel_regularizer=keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv1'
        )(inputs)
        x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
        x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_pool1')(x)  # 200 → 100
        
        # Second conv block: higher-level feature combinations
        x = layers.Conv1D(
            filters=64,
            kernel_size=3,
            activation='relu',
            padding='same',
            kernel_regularizer=keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv2'
        )(x)
        x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
        x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_pool2')(x)  # 100 → 50
        x = layers.Dropout(0.2, name=f'{name_prefix}_drop1')(x)
        
        return x  # Shape: (batch, 50, 64)
    
    def _lmu_temporal_block(self, inputs, memory_d=4, order=64, theta=100.0, 
                            hidden_units=128, name_prefix='lmu'):
        """
        LMU temporal modeling block.
        
        Uses Legendre Memory Units to capture temporal dynamics.
        The LMU's continuous-time memory representation is more efficient
        than LSTM gating for embedded deployment.
        
        Args:
            inputs: tensor of shape (batch, timesteps, features)
            memory_d: memory dimension per input feature
            order: Legendre polynomial order (temporal resolution)
            theta: time constant (window length the LMU "remembers")
            hidden_units: size of the hidden processing cell
        
        Returns: feature tensor of shape (batch, hidden_units)
        """
        x = LMU(
            memory_d=memory_d,
            order=order,
            theta=theta,
            hidden_cell=layers.LSTMCell(hidden_units),
            kernel_regularizer=keras.regularizers.l2(1e-5),
            dropout=0.2,
            name=f'{name_prefix}_layer'
        )(inputs)
        
        return x  # Shape: (batch, hidden_units)
    
    def _classification_head(self, features, hidden_dims=[128, 64], 
                             dropout_rate=0.3, name_prefix='head'):
        """
        Shared classification head.
        
        Args:
            features: flattened feature tensor
            hidden_dims: list of dense layer sizes
            dropout_rate: dropout rate between dense layers
        
        Returns: softmax output tensor of shape (batch, n_classes)
        """
        x = features
        for i, dim in enumerate(hidden_dims):
            x = layers.Dense(dim, activation='relu', name=f'{name_prefix}_dense{i+1}')(x)
            x = layers.BatchNormalization(name=f'{name_prefix}_bn{i+1}')(x)
            x = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop{i+1}')(x)
        
        output = layers.Dense(
            self.n_classes,
            activation='softmax',
            name=f'{name_prefix}_output'
        )(x)
        
        return output
    
    # =========================================================================
    # Model Variants
    # =========================================================================
    
    def build_cnn_only(self):
        """
        CNN-only model — spatial features with global pooling.
        Embedded baseline: smallest model, fastest inference.
        
        Architecture:
            Input(200,6) → Conv1D(32,k5) → Pool → Conv1D(64,k3) → Pool 
            → GlobalAvgPool → Dense(128) → Dense(64) → Dense(6)
        """
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        # CNN feature extraction
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        
        # Collapse temporal dimension — no sequence modeling
        x = layers.GlobalAveragePooling1D(name='global_pool')(cnn_features)
        
        # Classification
        output = self._classification_head(x, hidden_dims=[128, 64], 
                                           name_prefix='cnn_head')
        
        self.model = models.Model(inputs=inputs, outputs=output, 
                                  name='FallNet_CNN_Only')
        return self.model
    
    def build_lmu_only(self):
        """
        LMU-only model — temporal modeling directly on raw IMU signals.
        Tests whether the LMU can learn both spatial and temporal features.
        
        Architecture:
            Input(200,6) → LMU(memory_d=2, order=64, hidden=128) 
            → Dense(512) → Dense(128) → Dense(6)
        """
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        # LMU directly on raw input
        lmu_features = self._lmu_temporal_block(
            inputs, 
            memory_d=2, 
            order=64, 
            theta=200.0,  # Full 1-second window at 200Hz
            hidden_units=128,
            name_prefix='lmu'
        )
        
        # Classification — wider first layer to compensate for no CNN preprocessing
        output = self._classification_head(lmu_features, hidden_dims=[512, 128], 
                                           name_prefix='lmu_head')
        
        self.model = models.Model(inputs=inputs, outputs=output, 
                                  name='FallNet_LMU_Only')
        return self.model
    
    def build_cnn_lmu_hybrid(self):
        """
        CNN→LMU Hybrid — the main contribution.
        
        CNN extracts spatial features from the 6-axis IMU channels,
        then LMU models the temporal dynamics over the CNN feature sequence.
        
        This separates concerns:
          - CNN learns: axis correlations, spike shapes, local patterns
          - LMU learns: temporal evolution, ADL→Fall transitions, phase dynamics
        
        Architecture:
            Input(200,6) → Conv1D(32,k5) → Pool → Conv1D(64,k3) → Pool
            → LMU(memory_d=4, order=64, hidden=128) 
            → Dense(128) → Dense(64) → Dense(6)
        
        The CNN reduces 200 timesteps to 50, so the LMU processes a 4x shorter
        sequence with 10x richer features (64 vs 6 channels). This is critical
        for embedded deployment — fewer LMU state updates = less compute.
        """
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        # Stage 1: CNN spatial feature extraction
        # Input: (batch, 200, 6) → Output: (batch, 50, 64)
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        
        # Stage 2: LMU temporal modeling over CNN features
        # Input: (batch, 50, 64) → Output: (batch, 128)
        # theta=50 because the sequence is now 50 timesteps, not 200
        lmu_features = self._lmu_temporal_block(
            cnn_features,
            memory_d=4,       # 4 memory dims per feature — richer temporal encoding
            order=64,         # Legendre polynomial order
            theta=50.0,       # Adjusted for the compressed 50-step sequence
            hidden_units=128, # Hidden state size
            name_prefix='lmu'
        )
        
        # Stage 3: Classification
        output = self._classification_head(lmu_features, hidden_dims=[128, 64], 
                                           name_prefix='hybrid_head')
        
        self.model = models.Model(inputs=inputs, outputs=output, 
                                  name='FallNet_CNN_LMU_Hybrid')
        return self.model
    
    def build_ensemble(self):
        """
        CNN+LMU Ensemble — parallel branches, averaged outputs.
        Original FallNet architecture adapted with LMU replacing LSTM.
        
        Architecture:
            Input(200,6) ─┬→ CNN branch  → softmax(6) ─┬→ Average → Output(6)
                          └→ LMU branch  → softmax(6) ─┘
        """
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        # CNN branch: spatial features → classification
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        cnn_pooled = layers.GlobalAveragePooling1D(name='cnn_global_pool')(cnn_features)
        cnn_output = self._classification_head(cnn_pooled, hidden_dims=[128, 64], 
                                               name_prefix='cnn_head')
        
        # LMU branch: temporal features → classification
        lmu_features = self._lmu_temporal_block(
            inputs, memory_d=2, order=64, theta=200.0, hidden_units=128,
            name_prefix='lmu'
        )
        lmu_output = self._classification_head(lmu_features, hidden_dims=[512, 128], 
                                               name_prefix='lmu_head')
        
        # Ensemble: average the two softmax outputs
        ensemble_output = layers.Average(name='ensemble_average')([cnn_output, lmu_output])
        
        self.model = models.Model(inputs=inputs, outputs=ensemble_output, 
                                  name='FallNet_CNN_LMU_Ensemble')
        return self.model
    
    # =========================================================================
    # Compilation
    # =========================================================================
    
    def compile_model(self, learning_rate=5e-4):
        """Compile model with standard training configuration."""
        if self.model is None:
            raise ValueError("Model not built yet. Call a build_* method first.")
        
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=[
                'accuracy',
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall')
            ]
        )
        
        return self.model
    
    def get_model_stats(self):
        """Print parameter counts and estimated size."""
        if self.model is None:
            raise ValueError("No model built yet.")
        
        trainable = np.sum([np.prod(v.shape) for v in self.model.trainable_weights])
        non_trainable = np.sum([np.prod(v.shape) for v in self.model.non_trainable_weights])
        total = trainable + non_trainable
        
        # Rough size estimates
        float32_size_mb = total * 4 / (1024 * 1024)
        int8_size_kb = total * 1 / 1024
        
        print(f"\n{'='*60}")
        print(f"MODEL: {self.model.name}")
        print(f"{'='*60}")
        print(f"Trainable params:     {trainable:>10,}")
        print(f"Non-trainable params: {non_trainable:>10,}")
        print(f"Total params:         {total:>10,}")
        print(f"Float32 size:         {float32_size_mb:>10.2f} MB")
        print(f"INT8 quantized (est): {int8_size_kb:>10.1f} KB")
        print(f"Nano BLE flash (1MB): {'✅ fits' if int8_size_kb < 500 else '⚠️  tight' if int8_size_kb < 900 else '❌ too large'}")
        print(f"{'='*60}")
        
        return {'trainable': trainable, 'non_trainable': non_trainable, 
                'total': total, 'float32_mb': float32_size_mb, 'int8_kb': int8_size_kb}

print("✅ FallNet class defined (4 variants: CNN-only, LMU-only, CNN→LMU Hybrid, Ensemble)")

# %% [markdown]
## 2. Build and Compare All Architectures

# %%
print("\n" + "="*80)
print("ARCHITECTURE COMPARISON")
print("="*80)

model_stats = {}

# Build each variant and print stats
for variant_name, build_fn in [
    ('CNN-only',       'build_cnn_only'),
    ('LMU-only',       'build_lmu_only'),
    ('CNN→LMU Hybrid', 'build_cnn_lmu_hybrid'),
    ('Ensemble',       'build_ensemble'),
]:
    with tf.device('/CPU:0'):
        fn = FallNet(input_shape=(200, 6), n_classes=6)
        getattr(fn, build_fn)()
        stats = fn.get_model_stats()
        model_stats[variant_name] = stats

# Summary table
print("\n" + "="*80)
print("PARAMETER COMPARISON SUMMARY")
print("="*80)
print(f"\n{'Model':<20s} {'Total Params':>14s} {'Float32 (MB)':>14s} {'INT8 (KB)':>12s} {'Nano Fit?':>10s}")
print("-" * 74)
for name, s in model_stats.items():
    fit = '✅' if s['int8_kb'] < 500 else '⚠️' if s['int8_kb'] < 900 else '❌'
    print(f"{name:<20s} {s['total']:>14,} {s['float32_mb']:>14.2f} {s['int8_kb']:>12.1f} {fit:>10s}")

# %% [markdown]
## 3. Select Model Variant for Training

# %%
# ===== CHOOSE YOUR MODEL VARIANT HERE =====
MODEL_VARIANT = 'hybrid'  # Options: 'cnn_only', 'lmu_only', 'hybrid', 'ensemble'

VARIANT_MAP = {
    'cnn_only':  'build_cnn_only',
    'lmu_only':  'build_lmu_only',
    'hybrid':    'build_cnn_lmu_hybrid',
    'ensemble':  'build_ensemble',
}

print(f"\n{'='*80}")
print(f"SELECTED MODEL: {MODEL_VARIANT.upper()}")
print(f"{'='*80}")

# Build the selected variant
with tf.device('/CPU:0'):
    fallnet = FallNet(input_shape=(200, 6), n_classes=6)
    model = getattr(fallnet, VARIANT_MAP[MODEL_VARIANT])()

model = fallnet.compile_model()
model.summary()
fallnet.get_model_stats()

# %% [markdown]
## 4. Training Configuration

# %%
BATCH_SIZE = 128
EPOCHS = 50
K_FOLDS = 5

print(f"\n{'='*80}")
print("TRAINING CONFIGURATION")
print(f"{'='*80}")
print(f"Model:      {MODEL_VARIANT}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {EPOCHS}")
print(f"K-Folds:    {K_FOLDS}")
print(f"Using data: {len(y_labels):,} samples, {len(np.unique(y_labels))} classes")

# %% [markdown]
## 5. Pre-Training Verification

# %%
print(f"\n{'='*80}")
print("PRE-TRAINING VERIFICATION")
print(f"{'='*80}")

print(f"✅ Data shapes:")
print(f"   X_data:        {X_data.shape}")
print(f"   y_labels:      {y_labels.shape}")
print(f"   y_categorical: {y_categorical.shape}")
print(f"\n✅ Classes: {len(np.unique(y_labels))} (should be 6)")
print(f"✅ Label range: {y_labels.min()}-{y_labels.max()} (should be 0-5)")
print(f"✅ Model output: {model.output_shape[-1]} (should be 6)")

assert X_data.shape[0] == y_labels.shape[0] == y_categorical.shape[0], "Shape mismatch!"
assert len(np.unique(y_labels)) == 6, "Should have 6 classes!"
assert y_labels.max() == 5, "Max label should be 5!"
assert model.output_shape[-1] == 6, "Model should output 6 classes!"

print("\n✅ All checks passed — ready to train!")

# %% [markdown]
## 6. K-Fold Cross-Validation Training

# %%
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_results = []
fold_histories = []

print(f"\n{'='*80}")
print(f"STARTING K-FOLD CROSS-VALIDATION — {MODEL_VARIANT.upper()}")
print(f"{'='*80}")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"{'='*80}")
    
    # Split data
    X_train, X_val = X_data[train_idx], X_data[val_idx]
    y_train, y_val = y_categorical[train_idx], y_categorical[val_idx]
    y_train_labels = y_labels[train_idx]
    
    print(f"Train: {X_train.shape[0]:,} samples | Val: {X_val.shape[0]:,} samples")
    
    # Build fresh model for this fold
    fallnet_fold = FallNet(input_shape=(200, 6), n_classes=6)
    model_fold = getattr(fallnet_fold, VARIANT_MAP[MODEL_VARIANT])()
    model_fold = fallnet_fold.compile_model()
    
    # Callbacks
    fold_callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=20,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=10,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=str(output_dir / f'fallnet_{MODEL_VARIANT}_fold_{fold}.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
    ]
    
    # Calculate class weights
    class_weights_array = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_labels),
        y=y_train_labels
    )
    class_weights = dict(enumerate(class_weights_array))
    
    # Surgical boosts for problem classes
    class_weights[0] *= 1.5  # Walking
    class_weights[3] *= 3.0  # Stumbles
    class_weights[4] *= 1.2  # Fall Initiation — safety critical
    
    # Cap to prevent instability
    MAX_WEIGHT = 5.0
    for k in class_weights:
        class_weights[k] = min(class_weights[k], MAX_WEIGHT)
    
    if fold == 1:  # Print weights only once
        print("\nClass Weights:")
        for cls_idx in range(6):
            print(f"  {reverse_label_map[cls_idx]:<30s}: {class_weights[cls_idx]:.2f}x")
    
    # Train
    print(f"\nTraining fold {fold}...")
    history = model_fold.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=fold_callbacks,
        verbose=1
    )
    
    # Evaluate
    val_loss, val_acc, val_precision, val_recall = model_fold.evaluate(
        X_val, y_val, batch_size=2, verbose=0
    )
    val_f1 = (2 * (val_precision * val_recall) / (val_precision + val_recall) 
              if (val_precision + val_recall) > 0 else 0)
    
    print(f"\n{'='*50}")
    print(f"Fold {fold} Results:")
    print(f"{'='*50}")
    print(f"Loss:      {val_loss:.4f}")
    print(f"Accuracy:  {val_acc:.4f}")
    print(f"Precision: {val_precision:.4f}")
    print(f"Recall:    {val_recall:.4f}")
    print(f"F1-Score:  {val_f1:.4f}")
    
    fold_results.append({
        'fold': fold,
        'val_loss': val_loss,
        'val_accuracy': val_acc,
        'val_precision': val_precision,
        'val_recall': val_recall,
        'val_f1': val_f1
    })
    
    fold_histories.append(history.history)
    print(f"✅ Model saved: fallnet_{MODEL_VARIANT}_fold_{fold}.keras")

print(f"\n{'='*80}")
print("K-FOLD CROSS-VALIDATION COMPLETE")
print(f"{'='*80}")

# %% [markdown]
## 7. Aggregate Results

# %%
results_df = pd.DataFrame(fold_results)

print(f"\n{'='*80}")
print("RESULTS ACROSS ALL FOLDS")
print(f"{'='*80}")
print(results_df.to_string(index=False))

mean_results = results_df.mean(numeric_only=True)
std_results = results_df.std(numeric_only=True)

print(f"\n{'='*80}")
print("AVERAGE PERFORMANCE ± STD")
print(f"{'='*80}")

metrics_table = []
for metric in ['val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1']:
    metrics_table.append({
        'Metric': metric,
        'Mean': f"{mean_results[metric]:.4f}",
        'Std': f"±{std_results[metric]:.4f}"
    })

metrics_df = pd.DataFrame(metrics_table)
print(metrics_df.to_string(index=False))

# %% [markdown]
## 8. Training History Visualization

# %%
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = [
    ('loss', 'Loss'),
    ('accuracy', 'Accuracy'),
    ('precision', 'Precision'),
    ('recall', 'Recall')
]

for idx, (metric, title) in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    for fold_num, history in enumerate(fold_histories, 1):
        epochs = range(1, len(history[metric]) + 1)
        ax.plot(epochs, history[metric], label=f'Fold {fold_num} Train', 
                alpha=0.5, linewidth=1)
        ax.plot(epochs, history[f'val_{metric}'], label=f'Fold {fold_num} Val',
                linestyle='--', alpha=0.7, linewidth=1.5)
    
    ax.set_title(f'{title} Across All Folds', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'FallNet {MODEL_VARIANT.upper()} Training History — 5-Fold CV',
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(output_dir / f'training_history_{MODEL_VARIANT}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Training history saved")

# %% [markdown]
## 9. Detailed Evaluation on Best Fold

# %%
best_fold = int(results_df.loc[results_df['val_f1'].idxmax(), 'fold'])

print(f"\n{'='*80}")
print(f"DETAILED EVALUATION — BEST FOLD #{best_fold}")
print(f"{'='*80}")
print(f"Best fold F1-Score: {results_df.loc[results_df['fold']==best_fold, 'val_f1'].values[0]:.4f}")

# Load best model
best_model = keras.models.load_model(
    output_dir / f'fallnet_{MODEL_VARIANT}_fold_{best_fold}.keras'
)

# Get predictions on ALL data
y_pred_probs = best_model.predict(X_data, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report
class_names = [reverse_label_map[i] for i in range(6)]

print(f"\n{'='*80}")
print(f"CLASSIFICATION REPORT — {MODEL_VARIANT.upper()} (Best Fold on All Data)")
print(f"{'='*80}")
print(classification_report(y_labels, y_pred, target_names=class_names, digits=4))

# %% [markdown]
## 10. Per-Class Detailed Metrics

# %%
print(f"\n{'='*80}")
print("PER-CLASS DETAILED METRICS")
print(f"{'='*80}")

print(f"\n{'Class':<40s} {'Precision':<12s} {'Recall':<12s} {'F1-Score':<12s} {'Support'}")
print("-" * 90)

for cls_idx in range(6):
    precision = precision_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    recall = recall_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    f1 = f1_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    support = np.sum(y_labels == cls_idx)
    
    print(f"{reverse_label_map[cls_idx]:<40s} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {support}")

# %% [markdown]
## 11. Confusion Matrix

# %%
cm = confusion_matrix(y_labels, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
plt.title(f'Confusion Matrix — {MODEL_VARIANT.upper()} (6 Classes)',
          fontsize=15, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(output_dir / f'confusion_matrix_{MODEL_VARIANT}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Confusion matrix saved")

# %% [markdown]
## 12. Final Summary

# %%
# Get Fall_Initiation metrics
fall_init_idx = label_map["Fall_Initiation"]
fall_init_precision = precision_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
fall_init_recall = recall_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
fall_init_f1 = f1_score(y_labels == fall_init_idx, y_pred == fall_init_idx)

# Get model stats
stats = fallnet.get_model_stats()

print(f"\n{'='*80}")
print("TRAINING COMPLETE — FINAL SUMMARY")
print(f"{'='*80}")

summary = f"""
✅ Successfully trained FallNet {MODEL_VARIANT.upper()} with 5-fold cross-validation

Configuration:
  - Model: {model.name} ({stats['total']:,} params)
  - INT8 estimated size: {stats['int8_kb']:.1f} KB
  - Total samples: {len(y_labels):,}
  - Training samples per fold: ~{len(y_labels)*0.8//K_FOLDS:,.0f}
  - Validation samples per fold: ~{len(y_labels)*0.2//K_FOLDS:,.0f}

Average Performance (5-fold CV):
  - Accuracy:  {mean_results['val_accuracy']:.4f} ± {std_results['val_accuracy']:.4f}
  - Precision: {mean_results['val_precision']:.4f} ± {std_results['val_precision']:.4f}
  - Recall:    {mean_results['val_recall']:.4f} ± {std_results['val_recall']:.4f}
  - F1-Score:  {mean_results['val_f1']:.4f} ± {std_results['val_f1']:.4f}

Fall_Initiation Performance (Critical Class):
  - Recall (Sensitivity): {fall_init_recall:.4f}
  - F1-Score:             {fall_init_f1:.4f}

Embedded Deployment Estimate:
  - Target: Arduino Nano 33 BLE Sense Rev2 (nRF52840)
  - Flash budget: 1 MB → Model uses ~{stats['int8_kb']:.0f} KB INT8
  - RAM budget: 256 KB → TFLite arena ~{stats['int8_kb']*0.3:.0f} KB estimated

Saved Files:
  - Training history:    training_history_{MODEL_VARIANT}.png
  - Confusion matrix:    confusion_matrix_{MODEL_VARIANT}.png
  - Best model:          fallnet_{MODEL_VARIANT}_fold_{best_fold}.keras
"""

print(summary)

with open(output_dir / f'training_summary_{MODEL_VARIANT}.txt', 'w') as f:
    f.write(summary)

print(f"✅ Summary saved to training_summary_{MODEL_VARIANT}.txt")

In [ ]:
# %% [markdown]
# # FallNet with Static-Unrolled LMU — TFLite Compatible
# 
# Replaces keras_lmu.LMU with a hand-written layer that statically unrolls
# the LMU recurrence. This eliminates tf.TensorListReserve / tf.while_loop
# ops that block TFLite conversion.
#
# The LMU math is identical — only the execution strategy changes:
#   keras_lmu: RNN(LMUCell) → dynamic tf.while_loop → TensorArrays → TFLite fails
#   StaticLMU: explicit Python for-loop at graph build time → static ops → TFLite works
#
# Drop-in replacement: just swap `from keras_lmu import LMU` for this file's StaticLMU.

# %%
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")


# %% [markdown]
## 1. Static LMU Layer (TFLite-Compatible)

# %%
class StaticLMUCell(layers.Layer):
    """
    Single-step LMU cell for manual unrolling.
    
    Implements the Legendre Memory Unit recurrence:
        m[t] = A @ m[t-1] + B @ x[t]     (memory state update via Legendre ODE)
        h[t] = hidden_cell([x[t]; m[t]])   (hidden state via LSTM/Dense)
    
    where A, B are the discretized Legendre matrices (fixed, not learned).
    
    This cell is called once per timestep in a Python for-loop,
    producing a static TF graph with no tf.while_loop or TensorArrays.
    """
    
    def __init__(self, input_dim, memory_d, order, theta, hidden_units, 
                 dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.input_dim = input_dim
        self.memory_d = memory_d
        self.order = order
        self.theta = theta
        self.hidden_units = hidden_units
        self.dropout_rate = dropout
        
        # Memory state size: memory_d dimensions * order polynomials
        self.memory_size = memory_d * order
        
    def build(self, input_shape):
        # Compute the discretized Legendre matrices A, B
        # These are FIXED (non-trainable) — derived from the Legendre ODE
        A, B = self._get_legendre_matrices(self.order, self.theta)
        
        # Store as non-trainable weights so they're saved with the model
        self.A = self.add_weight(
            name='A', shape=(self.order, self.order),
            initializer=keras.initializers.Constant(A),
            trainable=False
        )
        self.B = self.add_weight(
            name='B', shape=(self.order, 1),
            initializer=keras.initializers.Constant(B),
            trainable=False
        )
        
        # Encoder: project input features to memory_d dimensions
        # Each of the memory_d dimensions gets its own projection of the input
        self.encoder = layers.Dense(
            self.memory_d,
            kernel_regularizer=keras.regularizers.l2(1e-5),
            name='lmu_encoder'
        )
        
        # Hidden cell: processes [input; flattened_memory] → hidden_state
        # Using Dense + tanh instead of LSTMCell for TFLite compatibility
        # (LSTMCell also uses internal state management that can cause issues)
        self.hidden_dense1 = layers.Dense(
            self.hidden_units,
            activation='tanh',
            kernel_regularizer=keras.regularizers.l2(1e-5),
            name='lmu_hidden1'
        )
        self.hidden_dense2 = layers.Dense(
            self.hidden_units,
            activation='tanh',
            kernel_regularizer=keras.regularizers.l2(1e-5),
            name='lmu_hidden2'
        )
        
        if self.dropout_rate > 0:
            self.dropout = layers.Dropout(self.dropout_rate)
        
        super().build(input_shape)
    
    @staticmethod
    def _get_legendre_matrices(order, theta):
        """
        Compute the discretized Legendre delay matrices A and B.
        
        These encode the continuous-time Legendre ODE:
            θ * dm/dt = A_cont @ m + B_cont @ x
        
        Discretized via Euler: m[t] = (I + A_cont/θ) @ m[t-1] + (B_cont/θ) @ x[t]
        
        The Legendre basis gives optimal polynomial approximation of the 
        input history over a sliding window of length θ timesteps.
        """
        Q = np.arange(order, dtype=np.float64)
        R = (2 * Q + 1)[:, None]
        j, i = np.meshgrid(Q, Q)
        
        # Continuous-time A matrix
        A = np.where(i < j, -1, (-1.0) ** (i - j + 1)) * R
        
        # Continuous-time B matrix
        B = (-1.0) ** Q[:, None] * R
        
        # Discretize (zero-order hold / Euler)
        C = np.eye(order) + A / theta
        D = B / theta
        
        return C.astype(np.float32), D.astype(np.float32)
    
    def call(self, x_t, memory_state, training=False):
        """
        Single timestep update.
        
        Args:
            x_t: input at time t, shape (batch, input_dim)
            memory_state: previous memory, shape (batch, memory_d, order)
            training: whether in training mode (for dropout)
            
        Returns:
            h_t: hidden output, shape (batch, hidden_units)
            new_memory: updated memory, shape (batch, memory_d, order)
        """
        # Encode input to memory_d dimensions: (batch, input_dim) → (batch, memory_d)
        u_t = self.encoder(x_t)  # (batch, memory_d)
        
        # Update memory state for each memory dimension
        # m[t] = A @ m[t-1] + B @ u[t]
        # memory_state: (batch, memory_d, order)
        # A: (order, order), B: (order, 1)
        
        # memory_state: (batch, memory_d, order)
        # A: (order, order)
        # For each memory dim, we need: new_m_d = m_d @ A^T  (vector-matrix product along order dim)
        # Batched: (batch, memory_d, order) @ (order, order) = (batch, memory_d, order)
        Am = tf.matmul(memory_state, self.A, transpose_b=True)  # (batch, memory_d, order)
        
        # B @ u_t: B is (order, 1), u_t is (batch, memory_d)
        # We want (batch, memory_d, order): each memory dim scaled by its u_t value
        # B squeezed: (order,) → broadcast with u_t: (batch, memory_d, 1) * (1, order)
        Bu = tf.expand_dims(u_t, axis=-1) * tf.reshape(self.B, [1, 1, self.order])
        # Bu shape: (batch, memory_d, order)
        
        new_memory = Am + Bu  # (batch, memory_d, order)
        
        # Flatten memory for hidden cell input
        m_flat = tf.reshape(new_memory, [-1, self.memory_size])  # (batch, memory_d * order)
        
        # Concatenate input with flattened memory
        h_input = tf.concat([x_t, m_flat], axis=-1)  # (batch, input_dim + memory_d * order)
        
        # Process through hidden layers
        h_t = self.hidden_dense1(h_input)
        if self.dropout_rate > 0:
            h_t = self.dropout(h_t, training=training)
        h_t = self.hidden_dense2(h_t)
        
        return h_t, new_memory


class StaticLMU(layers.Layer):
    """
    TFLite-compatible LMU layer with static unrolling.
    
    Instead of using tf.keras.layers.RNN (which creates dynamic TensorArrays),
    this layer unrolls the LMU recurrence in a Python for-loop at graph 
    construction time. The resulting TF graph has only static ops.
    
    The sequence length MUST be known at build time (which it is for our
    CNN→LMU hybrid: CNN always outputs exactly 50 timesteps).
    
    Usage:
        # Replace:
        #   from keras_lmu import LMU
        #   x = LMU(memory_d=4, order=64, theta=50, ...)(inputs)
        # With:
        x = StaticLMU(memory_d=4, order=64, theta=50, hidden_units=128)(inputs)
    """
    
    def __init__(self, memory_d, order, theta, hidden_units, dropout=0.0, 
                 return_sequences=False, **kwargs):
        super().__init__(**kwargs)
        self.memory_d = memory_d
        self.order = order
        self.theta = theta
        self.hidden_units = hidden_units
        self.dropout_rate = dropout
        self.return_sequences = return_sequences
        
    def build(self, input_shape):
        # input_shape: (batch, timesteps, features)
        self.timesteps = input_shape[1]
        self.input_dim = input_shape[2]
        
        if self.timesteps is None:
            raise ValueError(
                "StaticLMU requires a known sequence length at build time. "
                "Got input shape with timesteps=None. "
                "Ensure the input tensor has a static time dimension."
            )
        
        self.cell = StaticLMUCell(
            input_dim=self.input_dim,
            memory_d=self.memory_d,
            order=self.order,
            theta=self.theta,
            hidden_units=self.hidden_units,
            dropout=self.dropout_rate,
            name='lmu_cell'
        )
        
        super().build(input_shape)
        
    def call(self, inputs, training=False):
        """
        Forward pass with static unrolling.
        
        Args:
            inputs: (batch, timesteps, features)
            training: whether in training mode
            
        Returns:
            If return_sequences=False: last hidden state (batch, hidden_units)
            If return_sequences=True: all hidden states (batch, timesteps, hidden_units)
        """
        batch_size = tf.shape(inputs)[0]
        
        # Initialize memory state to zeros
        memory = tf.zeros([batch_size, self.memory_d, self.order])
        
        # Split input along time axis — this creates self.timesteps separate tensors
        # Each is (batch, features), and the list has a known Python length
        x_steps = tf.unstack(inputs, num=self.timesteps, axis=1)
        
        if self.return_sequences:
            outputs = []
        
        # Static unroll: Python for-loop generates fixed graph ops
        # No tf.while_loop, no TensorArrays, no dynamic shapes
        for t in range(self.timesteps):
            h_t, memory = self.cell(x_steps[t], memory, training=training)
            if self.return_sequences:
                outputs.append(h_t)
        
        if self.return_sequences:
            return tf.stack(outputs, axis=1)  # (batch, timesteps, hidden_units)
        else:
            return h_t  # (batch, hidden_units) — last timestep only
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'memory_d': self.memory_d,
            'order': self.order,
            'theta': self.theta,
            'hidden_units': self.hidden_units,
            'dropout': self.dropout_rate,
            'return_sequences': self.return_sequences,
        })
        return config


print("✅ StaticLMU layer defined (TFLite-compatible, static unrolling)")


# %% [markdown]
## 2. FallNet with StaticLMU

# %%
class FallNet:
    """
    FallNet: CNN-LMU Architectures for Pre-Impact Fall Detection
    
    Now using StaticLMU instead of keras_lmu.LMU for TFLite compatibility.
    
    Supports 4 model variants for ablation study:
      1. CNN-only         — spatial features only (embedded baseline)
      2. LMU-only         — temporal modeling on raw input (StaticLMU)
      3. CNN→LMU Hybrid   — CNN spatial extraction → StaticLMU temporal backbone
      4. CNN+LMU Ensemble — parallel branches with averaged outputs
    """
    
    def __init__(self, input_shape=(200, 6), n_classes=6):
        self.input_shape = input_shape
        self.n_classes = n_classes
        self.model = None
    
    # =========================================================================
    # Building Blocks
    # =========================================================================
    
    def _cnn_feature_extractor(self, inputs, name_prefix='cnn'):
        """
        CNN spatial feature extractor.
        Input: (batch, 200, 6) → Output: (batch, 50, 64)
        """
        x = layers.Conv1D(
            filters=32, kernel_size=5, activation='relu', padding='same',
            kernel_regularizer=keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv1'
        )(inputs)
        x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
        x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_pool1')(x)
        
        x = layers.Conv1D(
            filters=64, kernel_size=3, activation='relu', padding='same',
            kernel_regularizer=keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv2'
        )(x)
        x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
        x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_pool2')(x)
        x = layers.Dropout(0.2, name=f'{name_prefix}_drop1')(x)
        
        return x
    
    def _lmu_temporal_block(self, inputs, memory_d=4, order=64, theta=100.0,
                            hidden_units=128, name_prefix='lmu'):
        """
        StaticLMU temporal modeling block (TFLite-compatible).
        Uses static unrolling — no dynamic ops.
        """
        x = StaticLMU(
            memory_d=memory_d,
            order=order,
            theta=theta,
            hidden_units=hidden_units,
            dropout=0.2,
            return_sequences=False,
            name=f'{name_prefix}_layer'
        )(inputs)
        
        return x
    
    def _classification_head(self, features, hidden_dims=[128, 64],
                             dropout_rate=0.3, name_prefix='head'):
        """Shared classification head."""
        x = features
        for i, dim in enumerate(hidden_dims):
            x = layers.Dense(dim, activation='relu', name=f'{name_prefix}_dense{i+1}')(x)
            x = layers.BatchNormalization(name=f'{name_prefix}_bn{i+1}')(x)
            x = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop{i+1}')(x)
        
        output = layers.Dense(
            self.n_classes, activation='softmax', name=f'{name_prefix}_output'
        )(x)
        return output
    
    # =========================================================================
    # Model Variants
    # =========================================================================
    
    def build_cnn_only(self):
        """CNN-only model — spatial features with global pooling."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        x = layers.GlobalAveragePooling1D(name='global_pool')(cnn_features)
        output = self._classification_head(x, hidden_dims=[128, 64], name_prefix='cnn_head')
        self.model = models.Model(inputs=inputs, outputs=output, name='FallNet_CNN_Only')
        return self.model
    
    def build_lmu_only(self):
        """LMU-only model — temporal modeling directly on raw IMU signals."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        lmu_features = self._lmu_temporal_block(
            inputs, memory_d=2, order=64, theta=200.0, hidden_units=128,
            name_prefix='lmu'
        )
        output = self._classification_head(lmu_features, hidden_dims=[512, 128],
                                           name_prefix='lmu_head')
        self.model = models.Model(inputs=inputs, outputs=output, name='FallNet_LMU_Only')
        return self.model
    
    def build_cnn_lmu_hybrid(self):
        """
        CNN→LMU Hybrid — the main contribution.
        CNN extracts spatial features, StaticLMU models temporal dynamics.
        theta=50 because CNN compresses 200→50 timesteps.
        """
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        
        lmu_features = self._lmu_temporal_block(
            cnn_features,
            memory_d=4, order=64, theta=50.0, hidden_units=128,
            name_prefix='lmu'
        )
        
        output = self._classification_head(lmu_features, hidden_dims=[128, 64],
                                           name_prefix='hybrid_head')
        
        self.model = models.Model(inputs=inputs, outputs=output,
                                  name='FallNet_CNN_LMU_Hybrid')
        return self.model
    
    def build_ensemble(self):
        """CNN+LMU Ensemble — parallel branches, averaged outputs."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        cnn_pooled = layers.GlobalAveragePooling1D(name='cnn_global_pool')(cnn_features)
        cnn_output = self._classification_head(cnn_pooled, hidden_dims=[128, 64],
                                               name_prefix='cnn_head')
        
        lmu_features = self._lmu_temporal_block(
            inputs, memory_d=2, order=64, theta=200.0, hidden_units=128,
            name_prefix='lmu'
        )
        lmu_output = self._classification_head(lmu_features, hidden_dims=[512, 128],
                                               name_prefix='lmu_head')
        
        ensemble_output = layers.Average(name='ensemble_average')([cnn_output, lmu_output])
        self.model = models.Model(inputs=inputs, outputs=ensemble_output,
                                  name='FallNet_CNN_LMU_Ensemble')
        return self.model
    
    # =========================================================================
    # Compilation & Stats
    # =========================================================================
    
    def compile_model(self, learning_rate=5e-4):
        if self.model is None:
            raise ValueError("Model not built yet.")
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy', keras.metrics.Precision(name='precision'),
                     keras.metrics.Recall(name='recall')]
        )
        return self.model
    
    def get_model_stats(self):
        if self.model is None:
            raise ValueError("No model built yet.")
        trainable = np.sum([np.prod(v.shape) for v in self.model.trainable_weights])
        non_trainable = np.sum([np.prod(v.shape) for v in self.model.non_trainable_weights])
        total = trainable + non_trainable
        float32_size_mb = total * 4 / (1024 * 1024)
        int8_size_kb = total * 1 / 1024
        
        print(f"\n{'='*60}")
        print(f"MODEL: {self.model.name}")
        print(f"{'='*60}")
        print(f"Trainable params:     {trainable:>10,}")
        print(f"Non-trainable params: {non_trainable:>10,}")
        print(f"Total params:         {total:>10,}")
        print(f"Float32 size:         {float32_size_mb:>10.2f} MB")
        print(f"INT8 quantized (est): {int8_size_kb:>10.1f} KB")
        print(f"Nano BLE flash (1MB): {'✅ fits' if int8_size_kb < 500 else '⚠️  tight' if int8_size_kb < 900 else '❌ too large'}")
        print(f"{'='*60}")
        
        return {'trainable': trainable, 'non_trainable': non_trainable,
                'total': total, 'float32_mb': float32_size_mb, 'int8_kb': int8_size_kb}

print("✅ FallNet class defined with StaticLMU (4 variants, TFLite-compatible)")


# %% [markdown]
## 3. Verify TFLite Conversion BEFORE Training
# Quick sanity check that the architecture actually converts.

# %%
print("\n" + "="*80)
print("TFLITE CONVERSION SMOKE TEST")
print("="*80)

with tf.device('/CPU:0'):
    test_fn = FallNet(input_shape=(200, 6), n_classes=6)
    test_model = test_fn.build_cnn_lmu_hybrid()
    test_fn.compile_model()
    test_fn.get_model_stats()

# Try converting
print("\nAttempting TFLite conversion...")
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(test_model)
    tflite_bytes = converter.convert()
    print(f"✅ Float32 TFLite conversion succeeded: {len(tflite_bytes)/1024:.1f} KB")
    
    # Try INT8 quantization too
    converter2 = tf.lite.TFLiteConverter.from_keras_model(test_model)
    converter2.optimizations = [tf.lite.Optimize.DEFAULT]
    
    def rep_data():
        for _ in range(50):
            yield [np.random.randn(1, 200, 6).astype(np.float32)]
    
    converter2.representative_dataset = rep_data
    converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter2.inference_input_type = tf.int8
    converter2.inference_output_type = tf.int8
    tflite_int8 = converter2.convert()
    print(f"✅ INT8 TFLite conversion succeeded: {len(tflite_int8)/1024:.1f} KB")
    
    print("\n🎯 Both quantization levels work — safe to train!")
    
except Exception as e:
    print(f"❌ TFLite conversion failed: {e}")
    print("\nDo NOT proceed with training — fix the layer first.")

del test_fn, test_model
print()


# %% [markdown]
## 4. Build and Compare All Architectures

# %%
print("\n" + "="*80)
print("ARCHITECTURE COMPARISON (StaticLMU)")
print("="*80)

model_stats = {}
for variant_name, build_fn in [
    ('CNN-only',       'build_cnn_only'),
    ('LMU-only',       'build_lmu_only'),
    ('CNN→LMU Hybrid', 'build_cnn_lmu_hybrid'),
    ('Ensemble',       'build_ensemble'),
]:
    with tf.device('/CPU:0'):
        fn = FallNet(input_shape=(200, 6), n_classes=6)
        getattr(fn, build_fn)()
        stats = fn.get_model_stats()
        model_stats[variant_name] = stats

print("\n" + "="*80)
print("PARAMETER COMPARISON SUMMARY")
print("="*80)
print(f"\n{'Model':<20s} {'Total Params':>14s} {'Float32 (MB)':>14s} {'INT8 (KB)':>12s} {'Nano Fit?':>10s}")
print("-" * 74)
for name, s in model_stats.items():
    fit = '✅' if s['int8_kb'] < 500 else '⚠️' if s['int8_kb'] < 900 else '❌'
    print(f"{name:<20s} {s['total']:>14,} {s['float32_mb']:>14.2f} {s['int8_kb']:>12.1f} {fit:>10s}")


# %% [markdown]
## 5. Select Model & Train

# %%
MODEL_VARIANT = 'hybrid'  # Options: 'cnn_only', 'lmu_only', 'hybrid', 'ensemble'

VARIANT_MAP = {
    'cnn_only':  'build_cnn_only',
    'lmu_only':  'build_lmu_only',
    'hybrid':    'build_cnn_lmu_hybrid',
    'ensemble':  'build_ensemble',
}

print(f"\n{'='*80}")
print(f"SELECTED MODEL: {MODEL_VARIANT.upper()}")
print(f"{'='*80}")

with tf.device('/CPU:0'):
    fallnet = FallNet(input_shape=(200, 6), n_classes=6)
    model = getattr(fallnet, VARIANT_MAP[MODEL_VARIANT])()

model = fallnet.compile_model()
model.summary()
fallnet.get_model_stats()


# %% [markdown]
## 6. Training Configuration

# %%
BATCH_SIZE = 128
EPOCHS = 50
K_FOLDS = 5

print(f"\n{'='*80}")
print("TRAINING CONFIGURATION")
print(f"{'='*80}")
print(f"Model:      {MODEL_VARIANT}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {EPOCHS}")
print(f"K-Folds:    {K_FOLDS}")
print(f"Using data: {len(y_labels):,} samples, {len(np.unique(y_labels))} classes")


# %% [markdown]
## 7. Pre-Training Verification

# %%
print(f"\n{'='*80}")
print("PRE-TRAINING VERIFICATION")
print(f"{'='*80}")
print(f"✅ Data shapes:")
print(f"   X_data:        {X_data.shape}")
print(f"   y_labels:      {y_labels.shape}")
print(f"   y_categorical: {y_categorical.shape}")
print(f"\n✅ Classes: {len(np.unique(y_labels))} (should be 6)")
print(f"✅ Label range: {y_labels.min()}-{y_labels.max()} (should be 0-5)")
print(f"✅ Model output: {model.output_shape[-1]} (should be 6)")

assert X_data.shape[0] == y_labels.shape[0] == y_categorical.shape[0], "Shape mismatch!"
assert len(np.unique(y_labels)) == 6, "Should have 6 classes!"
assert y_labels.max() == 5, "Max label should be 5!"
assert model.output_shape[-1] == 6, "Model should output 6 classes!"

print("\n✅ All checks passed — ready to train!")


# %% [markdown]
## 8. K-Fold Cross-Validation Training

# %%
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_results = []
fold_histories = []

print(f"\n{'='*80}")
print(f"STARTING K-FOLD CROSS-VALIDATION — {MODEL_VARIANT.upper()} (StaticLMU)")
print(f"{'='*80}")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"{'='*80}")
    
    X_train, X_val = X_data[train_idx], X_data[val_idx]
    y_train, y_val = y_categorical[train_idx], y_categorical[val_idx]
    y_train_labels = y_labels[train_idx]
    
    print(f"Train: {X_train.shape[0]:,} samples | Val: {X_val.shape[0]:,} samples")
    
    # Build fresh model
    fallnet_fold = FallNet(input_shape=(200, 6), n_classes=6)
    model_fold = getattr(fallnet_fold, VARIANT_MAP[MODEL_VARIANT])()
    model_fold = fallnet_fold.compile_model()
    
    fold_callbacks = [
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1),
        ModelCheckpoint(
            filepath=str(output_dir / f'fallnet_{MODEL_VARIANT}_staticlmu_fold_{fold}.keras'),
            monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
        )
    ]
    
    # Class weights
    class_weights_array = compute_class_weight(
        class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels
    )
    class_weights = dict(enumerate(class_weights_array))
    class_weights[0] *= 1.5   # Walking
    class_weights[3] *= 3.0   # Stumbles
    class_weights[4] *= 1.2   # Fall Initiation
    MAX_WEIGHT = 5.0
    for k in class_weights:
        class_weights[k] = min(class_weights[k], MAX_WEIGHT)
    
    if fold == 1:
        print("\nClass Weights:")
        for cls_idx in range(6):
            print(f"  {reverse_label_map[cls_idx]:<30s}: {class_weights[cls_idx]:.2f}x")
    
    print(f"\nTraining fold {fold}...")
    history = model_fold.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=fold_callbacks,
        verbose=1
    )
    
    val_loss, val_acc, val_precision, val_recall = model_fold.evaluate(
        X_val, y_val, batch_size=2, verbose=0
    )
    val_f1 = (2 * (val_precision * val_recall) / (val_precision + val_recall)
              if (val_precision + val_recall) > 0 else 0)
    
    print(f"\n{'='*50}")
    print(f"Fold {fold} Results:")
    print(f"{'='*50}")
    print(f"Loss:      {val_loss:.4f}")
    print(f"Accuracy:  {val_acc:.4f}")
    print(f"Precision: {val_precision:.4f}")
    print(f"Recall:    {val_recall:.4f}")
    print(f"F1-Score:  {val_f1:.4f}")
    
    fold_results.append({
        'fold': fold, 'val_loss': val_loss, 'val_accuracy': val_acc,
        'val_precision': val_precision, 'val_recall': val_recall, 'val_f1': val_f1
    })
    fold_histories.append(history.history)
    print(f"✅ Model saved: fallnet_{MODEL_VARIANT}_staticlmu_fold_{fold}.keras")

print(f"\n{'='*80}")
print("K-FOLD CROSS-VALIDATION COMPLETE")
print(f"{'='*80}")


# %% [markdown]
## 9. Results & Evaluation (same as before)

# %%
results_df = pd.DataFrame(fold_results)

print(f"\n{'='*80}")
print("RESULTS ACROSS ALL FOLDS")
print(f"{'='*80}")
print(results_df.to_string(index=False))

mean_results = results_df.mean(numeric_only=True)
std_results = results_df.std(numeric_only=True)

print(f"\n{'='*80}")
print("AVERAGE PERFORMANCE ± STD")
print(f"{'='*80}")
for metric in ['val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1']:
    print(f"  {metric:<16s}: {mean_results[metric]:.4f} ±{std_results[metric]:.4f}")

# Training history visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
metrics = [('loss', 'Loss'), ('accuracy', 'Accuracy'), ('precision', 'Precision'), ('recall', 'Recall')]
for idx, (metric, title) in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    for fold_num, history in enumerate(fold_histories, 1):
        epochs = range(1, len(history[metric]) + 1)
        ax.plot(epochs, history[metric], label=f'Fold {fold_num} Train', alpha=0.5, linewidth=1)
        ax.plot(epochs, history[f'val_{metric}'], label=f'Fold {fold_num} Val',
                linestyle='--', alpha=0.7, linewidth=1.5)
    ax.set_title(f'{title} Across All Folds', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle(f'FallNet {MODEL_VARIANT.upper()} (StaticLMU) — 5-Fold CV',
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(output_dir / f'training_history_{MODEL_VARIANT}_staticlmu.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training history saved")


# %% [markdown]
## 10. Best Fold Evaluation

# %%
best_fold = int(results_df.loc[results_df['val_f1'].idxmax(), 'fold'])
print(f"\n{'='*80}")
print(f"DETAILED EVALUATION — BEST FOLD #{best_fold}")
print(f"{'='*80}")

best_model = keras.models.load_model(
    output_dir / f'fallnet_{MODEL_VARIANT}_staticlmu_fold_{best_fold}.keras'
)

y_pred_probs = best_model.predict(X_data, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
class_names = [reverse_label_map[i] for i in range(6)]

print(f"\n{'='*80}")
print(f"CLASSIFICATION REPORT — {MODEL_VARIANT.upper()} StaticLMU")
print(f"{'='*80}")
print(classification_report(y_labels, y_pred, target_names=class_names, digits=4))

# Per-class metrics
print(f"\n{'Class':<40s} {'Precision':<12s} {'Recall':<12s} {'F1-Score':<12s} {'Support'}")
print("-" * 90)
for cls_idx in range(6):
    p = precision_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    r = recall_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    f = f1_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    s = np.sum(y_labels == cls_idx)
    print(f"{reverse_label_map[cls_idx]:<40s} {p:<12.4f} {r:<12.4f} {f:<12.4f} {s}")

# Confusion matrix
cm = confusion_matrix(y_labels, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix — {MODEL_VARIANT.upper()} StaticLMU', fontsize=15, fontweight='bold', pad=20)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(output_dir / f'confusion_matrix_{MODEL_VARIANT}_staticlmu.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix saved")


# %% [markdown]
## 11. TFLite Quantization — All Variants

# %%
print(f"\n{'='*80}")
print("TFLITE QUANTIZATION — TRAINED MODEL")
print(f"{'='*80}")

# Fall_Initiation metrics
fall_init_idx = label_map["Fall_Initiation"]
keras_fi_recall = recall_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
keras_fi_f1 = f1_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
keras_accuracy = np.mean(y_pred == y_labels)

print(f"Keras baseline: Accuracy={keras_accuracy:.4f}, FI_Recall={keras_fi_recall:.4f}")

def representative_dataset_gen():
    indices = np.random.choice(len(X_data), size=min(500, len(X_data)), replace=False)
    for i in indices:
        yield [X_data[i:i+1].astype(np.float32)]

def evaluate_tflite(tflite_model, X_test, y_test):
    """Run TFLite inference and evaluate."""
    import time
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_dtype = input_details[0]['dtype']
    
    input_scale = input_details[0].get('quantization_parameters', {}).get('scales', [1.0])
    input_zp = input_details[0].get('quantization_parameters', {}).get('zero_points', [0])
    output_scale = output_details[0].get('quantization_parameters', {}).get('scales', [1.0])
    output_zp = output_details[0].get('quantization_parameters', {}).get('zero_points', [0])
    
    if len(input_scale) > 0: input_scale = input_scale[0]
    if len(input_zp) > 0: input_zp = input_zp[0]
    if len(output_scale) > 0: output_scale = output_scale[0]
    if len(output_zp) > 0: output_zp = output_zp[0]
    
    predictions = []
    inference_times = []
    
    for i in range(len(X_test)):
        sample = X_test[i:i+1].astype(np.float32)
        if input_dtype == np.int8:
            sample = (sample / input_scale + input_zp).astype(np.int8)
        elif input_dtype == np.uint8:
            sample = (sample / input_scale + input_zp).astype(np.uint8)
        
        interpreter.set_tensor(input_details[0]['index'], sample)
        start = time.perf_counter()
        interpreter.invoke()
        elapsed = time.perf_counter() - start
        inference_times.append(elapsed)
        
        output = interpreter.get_tensor(output_details[0]['index'])
        if output_details[0]['dtype'] in [np.int8, np.uint8]:
            output = (output.astype(np.float32) - output_zp) * output_scale
        predictions.append(np.argmax(output, axis=1)[0])
    
    return np.array(predictions), np.array(inference_times)

quant_configs = [
    ('float32',      'none'),
    ('float16',      'float16'),
    ('int8_dynamic', 'int8_weights'),
    ('int8_full',    'int8_full'),
]

quant_results = {}

for name, mode in quant_configs:
    print(f"\n{'─'*60}")
    print(f"Converting: {name}")
    print(f"{'─'*60}")
    
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
        
        if mode == 'float16':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.target_spec.supported_types = [tf.float16]
        elif mode == 'int8_weights':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
        elif mode == 'int8_full':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.representative_dataset = representative_dataset_gen
            converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            converter.inference_input_type = tf.int8
            converter.inference_output_type = tf.int8
        
        tflite_model = converter.convert()
        
        tflite_path = output_dir / f'fallnet_hybrid_staticlmu_{name}.tflite'
        with open(tflite_path, 'wb') as f:
            f.write(tflite_model)
        
        size_kb = len(tflite_model) / 1024
        print(f"  Size: {size_kb:.1f} KB")
        print(f"  Evaluating on {len(X_data):,} samples...")
        
        y_pred_tflite, inf_times = evaluate_tflite(tflite_model, X_data, y_labels)
        
        fi_recall = recall_score(y_labels == fall_init_idx, y_pred_tflite == fall_init_idx)
        fi_f1 = f1_score(y_labels == fall_init_idx, y_pred_tflite == fall_init_idx)
        acc = np.mean(y_pred_tflite == y_labels)
        
        quant_results[name] = {
            'size_kb': size_kb, 'accuracy': acc,
            'fall_init_recall': fi_recall, 'fall_init_f1': fi_f1,
            'mean_inference_us': np.mean(inf_times) * 1e6,
            'tflite_path': str(tflite_path),
        }
        
        print(f"  Accuracy:         {acc:.4f} ({(keras_accuracy - acc)*100:+.2f}%)")
        print(f"  Fall_Init Recall: {fi_recall:.4f} ({(keras_fi_recall - fi_recall)*100:+.2f}%)")
        print(f"  Fall_Init F1:     {fi_f1:.4f}")
        print(f"  ✅ Saved: {tflite_path.name}")
        
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        quant_results[name] = {'error': str(e)}


# %% [markdown]
## 12. Quantization Comparison

# %%
print(f"\n{'='*80}")
print("QUANTIZATION COMPARISON TABLE")
print(f"{'='*80}")

print(f"\n{'Variant':<16s} {'Size (KB)':>10s} {'Accuracy':>10s} {'FI Recall':>10s} {'FI F1':>10s} {'Acc Drop':>10s} {'Compression':>12s}")
print("-" * 82)

valid_results = {k: v for k, v in quant_results.items() if 'error' not in v}
base_size = list(valid_results.values())[0]['size_kb'] if valid_results else 1

for name, r in valid_results.items():
    print(f"{name:<16s} {r['size_kb']:>10.1f} {r['accuracy']:>10.4f} {r['fall_init_recall']:>10.4f} "
          f"{r['fall_init_f1']:>10.4f} {(keras_accuracy - r['accuracy'])*100:>+9.2f}% {base_size/r['size_kb']:>11.1f}x")

# Budget check
print(f"\n{'─'*60}")
print("NANO BLE SENSE REV2 BUDGET CHECK")
print(f"{'─'*60}")
for name, r in valid_results.items():
    flash_pct = r['size_kb'] / 600 * 100
    fit = '✅' if r['size_kb'] < 400 else '⚠️' if r['size_kb'] < 600 else '❌'
    print(f"  {name:<16s}: {r['size_kb']:>7.1f} KB = {flash_pct:>5.1f}% of ~600KB available  {fit}")


# %% [markdown]
## 13. Generate C Header for Deployment

# %%
# Pick best quantized model: smallest that keeps >95% FI recall
best_quant = None
for name in ['int8_full', 'int8_dynamic', 'float16', 'float32']:
    if name in valid_results and valid_results[name]['fall_init_recall'] >= 0.95:
        best_quant = name
        break

if best_quant is None:
    best_quant = list(valid_results.keys())[0]

print(f"\n{'='*80}")
print(f"DEPLOYMENT MODEL: {best_quant.upper()}")
print(f"{'='*80}")

r = valid_results[best_quant]
print(f"Size:             {r['size_kb']:.1f} KB")
print(f"Accuracy:         {r['accuracy']:.4f}")
print(f"Fall_Init Recall: {r['fall_init_recall']:.4f}")

# Generate C header
tflite_path = output_dir / f'fallnet_hybrid_staticlmu_{best_quant}.tflite'
header_path = output_dir / 'fallnet_model.h'

with open(tflite_path, 'rb') as f:
    data = f.read()

with open(header_path, 'w') as f:
    f.write(f'// FallNet CNN→LMU Hybrid (StaticLMU) — {best_quant} quantized\n')
    f.write(f'// Model size: {len(data)} bytes ({len(data)/1024:.1f} KB)\n')
    f.write(f'// Fall_Initiation Recall: {r["fall_init_recall"]:.4f}\n')
    f.write(f'// Overall Accuracy: {r["accuracy"]:.4f}\n')
    f.write(f'// Target: Arduino Nano 33 BLE Sense Rev2 (nRF52840)\n\n')
    f.write(f'#ifndef FALLNET_MODEL_H\n')
    f.write(f'#define FALLNET_MODEL_H\n\n')
    f.write(f'const unsigned int fallnet_model_len = {len(data)};\n')
    f.write(f'alignas(8) const unsigned char fallnet_model[] = {{\n')
    for i in range(0, len(data), 12):
        chunk = data[i:i+12]
        hex_vals = ', '.join(f'0x{b:02x}' for b in chunk)
        f.write(f'  {hex_vals},\n')
    f.write(f'}};\n\n')
    f.write(f'#endif // FALLNET_MODEL_H\n')

print(f"✅ C header saved: {header_path}")
print(f"   Array: fallnet_model[{len(data)}] ({len(data)/1024:.1f} KB)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
names = list(valid_results.keys())
sizes = [valid_results[n]['size_kb'] for n in names]
accs = [valid_results[n]['accuracy'] * 100 for n in names]
fi_recalls = [valid_results[n]['fall_init_recall'] * 100 for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336'][:len(names)]

ax1 = axes[0]
bars = ax1.bar(names, sizes, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Model Size (KB)', fontweight='bold')
ax1.set_title('Model Size by Quantization', fontweight='bold')
ax1.axhline(y=600, color='red', linestyle='--', alpha=0.7, label='Flash budget')
ax1.legend()
ax1_twin = ax1.twinx()
ax1_twin.plot(names, accs, 'ko-', markersize=8, linewidth=2, label='Accuracy %')
ax1_twin.set_ylabel('Accuracy (%)', fontweight='bold')
ax1_twin.legend(loc='center right')

ax2 = axes[1]
ax2.bar(names, fi_recalls, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Fall_Initiation Recall (%)', fontweight='bold')
ax2.set_title('Safety Metric by Quantization', fontweight='bold')
ax2.axhline(y=95, color='red', linestyle='--', alpha=0.7, label='95% threshold')
ax2.axhline(y=99, color='green', linestyle='--', alpha=0.5, label='99% target')
ax2.set_ylim(min(fi_recalls) - 3, 101)
ax2.legend()
for i, r in enumerate(fi_recalls):
    ax2.text(i, r + 0.3, f'{r:.1f}%', ha='center', fontweight='bold')

plt.suptitle('FallNet CNN→LMU (StaticLMU) — Quantization Tradeoffs', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / 'quantization_comparison_staticlmu.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Quantization comparison chart saved")


# %% [markdown]
## 14. Final Summary

# %%
stats = fallnet.get_model_stats()

print(f"\n{'='*80}")
print("TRAINING + QUANTIZATION COMPLETE — FINAL SUMMARY")
print(f"{'='*80}")

summary = f"""
✅ FallNet {MODEL_VARIANT.upper()} with StaticLMU — 5-fold CV + Quantization

Architecture: CNN→StaticLMU Hybrid (TFLite-compatible, static unrolling)
  - StaticLMU replaces keras_lmu.LMU (eliminates TensorListReserve ops)
  - Identical LMU math (Legendre memory), different execution strategy

Model: {stats['total']:,} params

Average Performance (5-fold CV):
  - Accuracy:  {mean_results['val_accuracy']:.4f} ± {std_results['val_accuracy']:.4f}
  - Precision: {mean_results['val_precision']:.4f} ± {std_results['val_precision']:.4f}
  - Recall:    {mean_results['val_recall']:.4f} ± {std_results['val_recall']:.4f}
  - F1-Score:  {mean_results['val_f1']:.4f} ± {std_results['val_f1']:.4f}

Fall_Initiation (Keras): Recall={keras_fi_recall:.4f}, F1={keras_fi_f1:.4f}

Quantization Results:
"""

for name, r in valid_results.items():
    summary += f"  {name:<16s}: {r['size_kb']:>7.1f} KB | Acc: {r['accuracy']:.4f} | FI Recall: {r['fall_init_recall']:.4f}\n"

summary += f"""
Deployment Model: {best_quant}
  Size: {valid_results[best_quant]['size_kb']:.1f} KB
  Accuracy: {valid_results[best_quant]['accuracy']:.4f}
  FI Recall: {valid_results[best_quant]['fall_init_recall']:.4f}

Target: Arduino Nano 33 BLE Sense Rev2 (nRF52840, 1MB flash, 256KB RAM)
Files: fallnet_hybrid_staticlmu_{best_quant}.tflite, fallnet_model.h
"""

print(summary)
with open(output_dir / f'training_summary_{MODEL_VARIANT}_staticlmu.txt', 'w') as f:
    f.write(summary)
print(f"✅ Summary saved")
print(f"\n🎯 Next: Flash fallnet_model.h to Nano BLE Sense, measure power with PPK2")

In [ ]:
# %% [markdown]
# # FallNet with Static-Unrolled LMU — TFLite Compatible
# 
# Replaces keras_lmu.LMU with a hand-written layer that statically unrolls
# the LMU recurrence. This eliminates tf.TensorListReserve / tf.while_loop
# ops that block TFLite conversion.
#
# The LMU math is identical — only the execution strategy changes:
#   keras_lmu: RNN(LMUCell) → dynamic tf.while_loop → TensorArrays → TFLite fails
#   StaticLMU: explicit Python for-loop at graph build time → static ops → TFLite works
#
# Drop-in replacement: just swap `from keras_lmu import LMU` for this file's StaticLMU.

# %%
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")


# %% [markdown]
## 1. Static LMU Layer (TFLite-Compatible)

# %%
class StaticLMUCell(layers.Layer):
    """
    Single-step LMU cell for manual unrolling.
    
    Implements the Legendre Memory Unit recurrence:
        m[t] = A @ m[t-1] + B @ x[t]     (memory state update via Legendre ODE)
        h[t] = hidden_cell([x[t]; m[t]])   (hidden state via LSTM/Dense)
    
    where A, B are the discretized Legendre matrices (fixed, not learned).
    
    This cell is called once per timestep in a Python for-loop,
    producing a static TF graph with no tf.while_loop or TensorArrays.
    """
    
    def __init__(self, input_dim, memory_d, order, theta, hidden_units, 
                 dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.input_dim = input_dim
        self.memory_d = memory_d
        self.order = order
        self.theta = theta
        self.hidden_units = hidden_units
        self.dropout_rate = dropout
        
        # Memory state size: memory_d dimensions * order polynomials
        self.memory_size = memory_d * order
        
    def build(self, input_shape):
        # Compute the discretized Legendre matrices A, B
        # These are FIXED (non-trainable) — derived from the Legendre ODE
        A, B = self._get_legendre_matrices(self.order, self.theta)
        
        # Store as non-trainable weights so they're saved with the model
        self.A = self.add_weight(
            name='A', shape=(self.order, self.order),
            initializer=keras.initializers.Constant(A),
            trainable=False
        )
        self.B = self.add_weight(
            name='B', shape=(self.order, 1),
            initializer=keras.initializers.Constant(B),
            trainable=False
        )
        
        # Encoder: project input features to memory_d dimensions
        # Each of the memory_d dimensions gets its own projection of the input
        self.encoder = layers.Dense(
            self.memory_d,
            kernel_regularizer=keras.regularizers.l2(1e-5),
            name='lmu_encoder'
        )
        
        # Hidden cell: processes [input; flattened_memory] → hidden_state
        # Using Dense + tanh instead of LSTMCell for TFLite compatibility
        # (LSTMCell also uses internal state management that can cause issues)
        self.hidden_dense1 = layers.Dense(
            self.hidden_units,
            activation='tanh',
            kernel_regularizer=keras.regularizers.l2(1e-5),
            name='lmu_hidden1'
        )
        self.hidden_dense2 = layers.Dense(
            self.hidden_units,
            activation='tanh',
            kernel_regularizer=keras.regularizers.l2(1e-5),
            name='lmu_hidden2'
        )
        
        if self.dropout_rate > 0:
            self.dropout = layers.Dropout(self.dropout_rate)
        
        super().build(input_shape)
    
    @staticmethod
    def _get_legendre_matrices(order, theta):
        """
        Compute the discretized Legendre delay matrices A and B.
        
        These encode the continuous-time Legendre ODE (HiPPO-LegS):
            θ * dm/dt = A_cont @ m + B_cont @ x
        
        Discretized via ZOH (zero-order hold) using matrix exponential:
            A_d = expm(A_cont / θ)
            B_d = A_cont^{-1} @ (A_d - I) @ B_cont
        
        This guarantees stability (all eigenvalues of A_d inside unit circle)
        unlike Euler discretization which diverges for high order/low theta.
        
        For order=64, theta=50:
            Euler:  max|eig| = 3.13 → UNSTABLE (NaN after ~10 steps)
            ZOH:    max|eig| = 0.80 → stable
        """
        from scipy.linalg import expm
        
        Q = np.arange(order, dtype=np.float64)
        R = (2 * Q + 1)[:, None]
        j, i = np.meshgrid(Q, Q)
        
        # Continuous-time matrices
        A_cont = np.where(i < j, -1, (-1.0) ** (i - j + 1)) * R
        B_cont = (-1.0) ** Q[:, None] * R
        
        # ZOH discretization via matrix exponential
        A_d = expm(A_cont / theta)
        
        # B_d = A_cont^{-1} @ (A_d - I) @ B_cont
        B_d = np.linalg.solve(A_cont, (A_d - np.eye(order)) @ B_cont)
        
        return A_d.astype(np.float32), B_d.astype(np.float32)
    
    def call(self, x_t, memory_state, training=False):
        """
        Single timestep update.
        
        Args:
            x_t: input at time t, shape (batch, input_dim)
            memory_state: previous memory, shape (batch, memory_d, order)
            training: whether in training mode (for dropout)
            
        Returns:
            h_t: hidden output, shape (batch, hidden_units)
            new_memory: updated memory, shape (batch, memory_d, order)
        """
        # Encode input to memory_d dimensions: (batch, input_dim) → (batch, memory_d)
        u_t = self.encoder(x_t)  # (batch, memory_d)
        
        # Update memory state for each memory dimension
        # m[t] = A @ m[t-1] + B @ u[t]
        # memory_state: (batch, memory_d, order)
        # A: (order, order), B: (order, 1)
        
        # memory_state: (batch, memory_d, order)
        # A: (order, order)
        # For each memory dim, we need: new_m_d = m_d @ A^T  (vector-matrix product along order dim)
        # Batched: (batch, memory_d, order) @ (order, order) = (batch, memory_d, order)
        Am = tf.matmul(memory_state, self.A, transpose_b=True)  # (batch, memory_d, order)
        
        # B @ u_t: B is (order, 1), u_t is (batch, memory_d)
        # We want (batch, memory_d, order): each memory dim scaled by its u_t value
        # B squeezed: (order,) → broadcast with u_t: (batch, memory_d, 1) * (1, order)
        Bu = tf.expand_dims(u_t, axis=-1) * tf.reshape(self.B, [1, 1, self.order])
        # Bu shape: (batch, memory_d, order)
        
        new_memory = Am + Bu  # (batch, memory_d, order)
        
        # Flatten memory for hidden cell input
        m_flat = tf.reshape(new_memory, [-1, self.memory_size])  # (batch, memory_d * order)
        
        # Concatenate input with flattened memory
        h_input = tf.concat([x_t, m_flat], axis=-1)  # (batch, input_dim + memory_d * order)
        
        # Process through hidden layers
        h_t = self.hidden_dense1(h_input)
        if self.dropout_rate > 0:
            h_t = self.dropout(h_t, training=training)
        h_t = self.hidden_dense2(h_t)
        
        return h_t, new_memory


class StaticLMU(layers.Layer):
    """
    TFLite-compatible LMU layer with static unrolling.
    
    Instead of using tf.keras.layers.RNN (which creates dynamic TensorArrays),
    this layer unrolls the LMU recurrence in a Python for-loop at graph 
    construction time. The resulting TF graph has only static ops.
    
    The sequence length MUST be known at build time (which it is for our
    CNN→LMU hybrid: CNN always outputs exactly 50 timesteps).
    
    Usage:
        # Replace:
        #   from keras_lmu import LMU
        #   x = LMU(memory_d=4, order=64, theta=50, ...)(inputs)
        # With:
        x = StaticLMU(memory_d=4, order=64, theta=50, hidden_units=128)(inputs)
    """
    
    def __init__(self, memory_d, order, theta, hidden_units, dropout=0.0, 
                 return_sequences=False, **kwargs):
        super().__init__(**kwargs)
        self.memory_d = memory_d
        self.order = order
        self.theta = theta
        self.hidden_units = hidden_units
        self.dropout_rate = dropout
        self.return_sequences = return_sequences
        
    def build(self, input_shape):
        # input_shape: (batch, timesteps, features)
        self.timesteps = input_shape[1]
        self.input_dim = input_shape[2]
        
        if self.timesteps is None:
            raise ValueError(
                "StaticLMU requires a known sequence length at build time. "
                "Got input shape with timesteps=None. "
                "Ensure the input tensor has a static time dimension."
            )
        
        self.cell = StaticLMUCell(
            input_dim=self.input_dim,
            memory_d=self.memory_d,
            order=self.order,
            theta=self.theta,
            hidden_units=self.hidden_units,
            dropout=self.dropout_rate,
            name='lmu_cell'
        )
        
        super().build(input_shape)
        
    def call(self, inputs, training=False):
        """
        Forward pass with static unrolling.
        
        Args:
            inputs: (batch, timesteps, features)
            training: whether in training mode
            
        Returns:
            If return_sequences=False: last hidden state (batch, hidden_units)
            If return_sequences=True: all hidden states (batch, timesteps, hidden_units)
        """
        batch_size = tf.shape(inputs)[0]
        
        # Initialize memory state to zeros
        memory = tf.zeros([batch_size, self.memory_d, self.order])
        
        # Split input along time axis — this creates self.timesteps separate tensors
        # Each is (batch, features), and the list has a known Python length
        x_steps = tf.unstack(inputs, num=self.timesteps, axis=1)
        
        if self.return_sequences:
            outputs = []
        
        # Static unroll: Python for-loop generates fixed graph ops
        # No tf.while_loop, no TensorArrays, no dynamic shapes
        for t in range(self.timesteps):
            h_t, memory = self.cell(x_steps[t], memory, training=training)
            if self.return_sequences:
                outputs.append(h_t)
        
        if self.return_sequences:
            return tf.stack(outputs, axis=1)  # (batch, timesteps, hidden_units)
        else:
            return h_t  # (batch, hidden_units) — last timestep only
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'memory_d': self.memory_d,
            'order': self.order,
            'theta': self.theta,
            'hidden_units': self.hidden_units,
            'dropout': self.dropout_rate,
            'return_sequences': self.return_sequences,
        })
        return config


print("✅ StaticLMU layer defined (TFLite-compatible, static unrolling)")


# %% [markdown]
## 2. FallNet with StaticLMU

# %%
class FallNet:
    """
    FallNet: CNN-LMU Architectures for Pre-Impact Fall Detection
    
    Now using StaticLMU instead of keras_lmu.LMU for TFLite compatibility.
    
    Supports 4 model variants for ablation study:
      1. CNN-only         — spatial features only (embedded baseline)
      2. LMU-only         — temporal modeling on raw input (StaticLMU)
      3. CNN→LMU Hybrid   — CNN spatial extraction → StaticLMU temporal backbone
      4. CNN+LMU Ensemble — parallel branches with averaged outputs
    """
    
    def __init__(self, input_shape=(200, 6), n_classes=6):
        self.input_shape = input_shape
        self.n_classes = n_classes
        self.model = None
    
    # =========================================================================
    # Building Blocks
    # =========================================================================
    
    def _cnn_feature_extractor(self, inputs, name_prefix='cnn'):
        """
        CNN spatial feature extractor.
        Input: (batch, 200, 6) → Output: (batch, 50, 64)
        """
        x = layers.Conv1D(
            filters=32, kernel_size=5, activation='relu', padding='same',
            kernel_regularizer=keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv1'
        )(inputs)
        x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
        x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_pool1')(x)
        
        x = layers.Conv1D(
            filters=64, kernel_size=3, activation='relu', padding='same',
            kernel_regularizer=keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv2'
        )(x)
        x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
        x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_pool2')(x)
        x = layers.Dropout(0.2, name=f'{name_prefix}_drop1')(x)
        
        return x
    
    def _lmu_temporal_block(self, inputs, memory_d=4, order=64, theta=100.0,
                            hidden_units=128, name_prefix='lmu'):
        """
        StaticLMU temporal modeling block (TFLite-compatible).
        Uses static unrolling — no dynamic ops.
        """
        x = StaticLMU(
            memory_d=memory_d,
            order=order,
            theta=theta,
            hidden_units=hidden_units,
            dropout=0.2,
            return_sequences=False,
            name=f'{name_prefix}_layer'
        )(inputs)
        
        return x
    
    def _classification_head(self, features, hidden_dims=[128, 64],
                             dropout_rate=0.3, name_prefix='head'):
        """Shared classification head."""
        x = features
        for i, dim in enumerate(hidden_dims):
            x = layers.Dense(dim, activation='relu', name=f'{name_prefix}_dense{i+1}')(x)
            x = layers.BatchNormalization(name=f'{name_prefix}_bn{i+1}')(x)
            x = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop{i+1}')(x)
        
        output = layers.Dense(
            self.n_classes, activation='softmax', name=f'{name_prefix}_output'
        )(x)
        return output
    
    # =========================================================================
    # Model Variants
    # =========================================================================
    
    def build_cnn_only(self):
        """CNN-only model — spatial features with global pooling."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        x = layers.GlobalAveragePooling1D(name='global_pool')(cnn_features)
        output = self._classification_head(x, hidden_dims=[128, 64], name_prefix='cnn_head')
        self.model = models.Model(inputs=inputs, outputs=output, name='FallNet_CNN_Only')
        return self.model
    
    def build_lmu_only(self):
        """LMU-only model — temporal modeling directly on raw IMU signals."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        lmu_features = self._lmu_temporal_block(
            inputs, memory_d=2, order=64, theta=200.0, hidden_units=128,
            name_prefix='lmu'
        )
        output = self._classification_head(lmu_features, hidden_dims=[512, 128],
                                           name_prefix='lmu_head')
        self.model = models.Model(inputs=inputs, outputs=output, name='FallNet_LMU_Only')
        return self.model
    
    def build_cnn_lmu_hybrid(self):
        """
        CNN→LMU Hybrid — the main contribution.
        CNN extracts spatial features, StaticLMU models temporal dynamics.
        theta=50 because CNN compresses 200→50 timesteps.
        """
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        
        lmu_features = self._lmu_temporal_block(
            cnn_features,
            memory_d=4, order=64, theta=50.0, hidden_units=128,
            name_prefix='lmu'
        )
        
        output = self._classification_head(lmu_features, hidden_dims=[128, 64],
                                           name_prefix='hybrid_head')
        
        self.model = models.Model(inputs=inputs, outputs=output,
                                  name='FallNet_CNN_LMU_Hybrid')
        return self.model
    
    def build_ensemble(self):
        """CNN+LMU Ensemble — parallel branches, averaged outputs."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        cnn_pooled = layers.GlobalAveragePooling1D(name='cnn_global_pool')(cnn_features)
        cnn_output = self._classification_head(cnn_pooled, hidden_dims=[128, 64],
                                               name_prefix='cnn_head')
        
        lmu_features = self._lmu_temporal_block(
            inputs, memory_d=2, order=64, theta=200.0, hidden_units=128,
            name_prefix='lmu'
        )
        lmu_output = self._classification_head(lmu_features, hidden_dims=[512, 128],
                                               name_prefix='lmu_head')
        
        ensemble_output = layers.Average(name='ensemble_average')([cnn_output, lmu_output])
        self.model = models.Model(inputs=inputs, outputs=ensemble_output,
                                  name='FallNet_CNN_LMU_Ensemble')
        return self.model
    
    # =========================================================================
    # Compilation & Stats
    # =========================================================================
    
    def compile_model(self, learning_rate=5e-4):
        if self.model is None:
            raise ValueError("Model not built yet.")
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy', keras.metrics.Precision(name='precision'),
                     keras.metrics.Recall(name='recall')]
        )
        return self.model
    
    def get_model_stats(self):
        if self.model is None:
            raise ValueError("No model built yet.")
        trainable = np.sum([np.prod(v.shape) for v in self.model.trainable_weights])
        non_trainable = np.sum([np.prod(v.shape) for v in self.model.non_trainable_weights])
        total = trainable + non_trainable
        float32_size_mb = total * 4 / (1024 * 1024)
        int8_size_kb = total * 1 / 1024
        
        print(f"\n{'='*60}")
        print(f"MODEL: {self.model.name}")
        print(f"{'='*60}")
        print(f"Trainable params:     {trainable:>10,}")
        print(f"Non-trainable params: {non_trainable:>10,}")
        print(f"Total params:         {total:>10,}")
        print(f"Float32 size:         {float32_size_mb:>10.2f} MB")
        print(f"INT8 quantized (est): {int8_size_kb:>10.1f} KB")
        print(f"Nano BLE flash (1MB): {'✅ fits' if int8_size_kb < 500 else '⚠️  tight' if int8_size_kb < 900 else '❌ too large'}")
        print(f"{'='*60}")
        
        return {'trainable': trainable, 'non_trainable': non_trainable,
                'total': total, 'float32_mb': float32_size_mb, 'int8_kb': int8_size_kb}

print("✅ FallNet class defined with StaticLMU (4 variants, TFLite-compatible)")


# %% [markdown]
## 3. Verify TFLite Conversion BEFORE Training
# Quick sanity check that the architecture actually converts.

# %%
print("\n" + "="*80)
print("TFLITE CONVERSION SMOKE TEST")
print("="*80)

with tf.device('/CPU:0'):
    test_fn = FallNet(input_shape=(200, 6), n_classes=6)
    test_model = test_fn.build_cnn_lmu_hybrid()
    test_fn.compile_model()
    test_fn.get_model_stats()

# Try converting
print("\nAttempting TFLite conversion...")
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(test_model)
    tflite_bytes = converter.convert()
    print(f"✅ Float32 TFLite conversion succeeded: {len(tflite_bytes)/1024:.1f} KB")
    
    # Try INT8 quantization too
    converter2 = tf.lite.TFLiteConverter.from_keras_model(test_model)
    converter2.optimizations = [tf.lite.Optimize.DEFAULT]
    
    def rep_data():
        for _ in range(50):
            yield [np.random.randn(1, 200, 6).astype(np.float32)]
    
    converter2.representative_dataset = rep_data
    converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter2.inference_input_type = tf.int8
    converter2.inference_output_type = tf.int8
    tflite_int8 = converter2.convert()
    print(f"✅ INT8 TFLite conversion succeeded: {len(tflite_int8)/1024:.1f} KB")
    
    print("\n🎯 Both quantization levels work — safe to train!")
    
except Exception as e:
    print(f"❌ TFLite conversion failed: {e}")
    print("\nDo NOT proceed with training — fix the layer first.")

del test_fn, test_model
print()


# %% [markdown]
## 4. Build and Compare All Architectures

# %%
print("\n" + "="*80)
print("ARCHITECTURE COMPARISON (StaticLMU)")
print("="*80)

model_stats = {}
for variant_name, build_fn in [
    ('CNN-only',       'build_cnn_only'),
    ('LMU-only',       'build_lmu_only'),
    ('CNN→LMU Hybrid', 'build_cnn_lmu_hybrid'),
    ('Ensemble',       'build_ensemble'),
]:
    with tf.device('/CPU:0'):
        fn = FallNet(input_shape=(200, 6), n_classes=6)
        getattr(fn, build_fn)()
        stats = fn.get_model_stats()
        model_stats[variant_name] = stats

print("\n" + "="*80)
print("PARAMETER COMPARISON SUMMARY")
print("="*80)
print(f"\n{'Model':<20s} {'Total Params':>14s} {'Float32 (MB)':>14s} {'INT8 (KB)':>12s} {'Nano Fit?':>10s}")
print("-" * 74)
for name, s in model_stats.items():
    fit = '✅' if s['int8_kb'] < 500 else '⚠️' if s['int8_kb'] < 900 else '❌'
    print(f"{name:<20s} {s['total']:>14,} {s['float32_mb']:>14.2f} {s['int8_kb']:>12.1f} {fit:>10s}")


# %% [markdown]
## 5. Select Model & Train

# %%
MODEL_VARIANT = 'hybrid'  # Options: 'cnn_only', 'lmu_only', 'hybrid', 'ensemble'

VARIANT_MAP = {
    'cnn_only':  'build_cnn_only',
    'lmu_only':  'build_lmu_only',
    'hybrid':    'build_cnn_lmu_hybrid',
    'ensemble':  'build_ensemble',
}

print(f"\n{'='*80}")
print(f"SELECTED MODEL: {MODEL_VARIANT.upper()}")
print(f"{'='*80}")

with tf.device('/CPU:0'):
    fallnet = FallNet(input_shape=(200, 6), n_classes=6)
    model = getattr(fallnet, VARIANT_MAP[MODEL_VARIANT])()

model = fallnet.compile_model()
model.summary()
fallnet.get_model_stats()


# %% [markdown]
## 6. Training Configuration

# %%
BATCH_SIZE = 128
EPOCHS = 50
K_FOLDS = 5

print(f"\n{'='*80}")
print("TRAINING CONFIGURATION")
print(f"{'='*80}")
print(f"Model:      {MODEL_VARIANT}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {EPOCHS}")
print(f"K-Folds:    {K_FOLDS}")
print(f"Using data: {len(y_labels):,} samples, {len(np.unique(y_labels))} classes")


# %% [markdown]
## 7. Pre-Training Verification

# %%
print(f"\n{'='*80}")
print("PRE-TRAINING VERIFICATION")
print(f"{'='*80}")
print(f"✅ Data shapes:")
print(f"   X_data:        {X_data.shape}")
print(f"   y_labels:      {y_labels.shape}")
print(f"   y_categorical: {y_categorical.shape}")
print(f"\n✅ Classes: {len(np.unique(y_labels))} (should be 6)")
print(f"✅ Label range: {y_labels.min()}-{y_labels.max()} (should be 0-5)")
print(f"✅ Model output: {model.output_shape[-1]} (should be 6)")

assert X_data.shape[0] == y_labels.shape[0] == y_categorical.shape[0], "Shape mismatch!"
assert len(np.unique(y_labels)) == 6, "Should have 6 classes!"
assert y_labels.max() == 5, "Max label should be 5!"
assert model.output_shape[-1] == 6, "Model should output 6 classes!"

print("\n✅ All checks passed — ready to train!")


# %% [markdown]
## 8. K-Fold Cross-Validation Training

# %%
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_results = []
fold_histories = []

print(f"\n{'='*80}")
print(f"STARTING K-FOLD CROSS-VALIDATION — {MODEL_VARIANT.upper()} (StaticLMU)")
print(f"{'='*80}")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"{'='*80}")
    
    X_train, X_val = X_data[train_idx], X_data[val_idx]
    y_train, y_val = y_categorical[train_idx], y_categorical[val_idx]
    y_train_labels = y_labels[train_idx]
    
    print(f"Train: {X_train.shape[0]:,} samples | Val: {X_val.shape[0]:,} samples")
    
    # Build fresh model
    fallnet_fold = FallNet(input_shape=(200, 6), n_classes=6)
    model_fold = getattr(fallnet_fold, VARIANT_MAP[MODEL_VARIANT])()
    model_fold = fallnet_fold.compile_model()
    
    fold_callbacks = [
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1),
        ModelCheckpoint(
            filepath=str(output_dir / f'fallnet_{MODEL_VARIANT}_staticlmu_fold_{fold}.keras'),
            monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
        )
    ]
    
    # Class weights
    class_weights_array = compute_class_weight(
        class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels
    )
    class_weights = dict(enumerate(class_weights_array))
    class_weights[0] *= 1.5   # Walking
    class_weights[3] *= 3.0   # Stumbles
    class_weights[4] *= 1.2   # Fall Initiation
    MAX_WEIGHT = 5.0
    for k in class_weights:
        class_weights[k] = min(class_weights[k], MAX_WEIGHT)
    
    if fold == 1:
        print("\nClass Weights:")
        for cls_idx in range(6):
            print(f"  {reverse_label_map[cls_idx]:<30s}: {class_weights[cls_idx]:.2f}x")
    
    print(f"\nTraining fold {fold}...")
    history = model_fold.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=fold_callbacks,
        verbose=1
    )
    
    val_loss, val_acc, val_precision, val_recall = model_fold.evaluate(
        X_val, y_val, batch_size=2, verbose=0
    )
    val_f1 = (2 * (val_precision * val_recall) / (val_precision + val_recall)
              if (val_precision + val_recall) > 0 else 0)
    
    print(f"\n{'='*50}")
    print(f"Fold {fold} Results:")
    print(f"{'='*50}")
    print(f"Loss:      {val_loss:.4f}")
    print(f"Accuracy:  {val_acc:.4f}")
    print(f"Precision: {val_precision:.4f}")
    print(f"Recall:    {val_recall:.4f}")
    print(f"F1-Score:  {val_f1:.4f}")
    
    fold_results.append({
        'fold': fold, 'val_loss': val_loss, 'val_accuracy': val_acc,
        'val_precision': val_precision, 'val_recall': val_recall, 'val_f1': val_f1
    })
    fold_histories.append(history.history)
    print(f"✅ Model saved: fallnet_{MODEL_VARIANT}_staticlmu_fold_{fold}.keras")

print(f"\n{'='*80}")
print("K-FOLD CROSS-VALIDATION COMPLETE")
print(f"{'='*80}")


# %% [markdown]
## 9. Results & Evaluation (same as before)

# %%
results_df = pd.DataFrame(fold_results)

print(f"\n{'='*80}")
print("RESULTS ACROSS ALL FOLDS")
print(f"{'='*80}")
print(results_df.to_string(index=False))

mean_results = results_df.mean(numeric_only=True)
std_results = results_df.std(numeric_only=True)

print(f"\n{'='*80}")
print("AVERAGE PERFORMANCE ± STD")
print(f"{'='*80}")
for metric in ['val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1']:
    print(f"  {metric:<16s}: {mean_results[metric]:.4f} ±{std_results[metric]:.4f}")

# Training history visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
metrics = [('loss', 'Loss'), ('accuracy', 'Accuracy'), ('precision', 'Precision'), ('recall', 'Recall')]
for idx, (metric, title) in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    for fold_num, history in enumerate(fold_histories, 1):
        epochs = range(1, len(history[metric]) + 1)
        ax.plot(epochs, history[metric], label=f'Fold {fold_num} Train', alpha=0.5, linewidth=1)
        ax.plot(epochs, history[f'val_{metric}'], label=f'Fold {fold_num} Val',
                linestyle='--', alpha=0.7, linewidth=1.5)
    ax.set_title(f'{title} Across All Folds', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle(f'FallNet {MODEL_VARIANT.upper()} (StaticLMU) — 5-Fold CV',
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(output_dir / f'training_history_{MODEL_VARIANT}_staticlmu.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training history saved")


# %% [markdown]
## 10. Best Fold Evaluation

# %%
best_fold = int(results_df.loc[results_df['val_f1'].idxmax(), 'fold'])
print(f"\n{'='*80}")
print(f"DETAILED EVALUATION — BEST FOLD #{best_fold}")
print(f"{'='*80}")

best_model = keras.models.load_model(
    output_dir / f'fallnet_{MODEL_VARIANT}_staticlmu_fold_{best_fold}.keras'
)

y_pred_probs = best_model.predict(X_data, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
class_names = [reverse_label_map[i] for i in range(6)]

print(f"\n{'='*80}")
print(f"CLASSIFICATION REPORT — {MODEL_VARIANT.upper()} StaticLMU")
print(f"{'='*80}")
print(classification_report(y_labels, y_pred, target_names=class_names, digits=4))

# Per-class metrics
print(f"\n{'Class':<40s} {'Precision':<12s} {'Recall':<12s} {'F1-Score':<12s} {'Support'}")
print("-" * 90)
for cls_idx in range(6):
    p = precision_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    r = recall_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    f = f1_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    s = np.sum(y_labels == cls_idx)
    print(f"{reverse_label_map[cls_idx]:<40s} {p:<12.4f} {r:<12.4f} {f:<12.4f} {s}")

# Confusion matrix
cm = confusion_matrix(y_labels, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix — {MODEL_VARIANT.upper()} StaticLMU', fontsize=15, fontweight='bold', pad=20)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(output_dir / f'confusion_matrix_{MODEL_VARIANT}_staticlmu.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix saved")


# %% [markdown]
## 11. TFLite Quantization — All Variants

# %%
print(f"\n{'='*80}")
print("TFLITE QUANTIZATION — TRAINED MODEL")
print(f"{'='*80}")

# Fall_Initiation metrics
fall_init_idx = label_map["Fall_Initiation"]
keras_fi_recall = recall_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
keras_fi_f1 = f1_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
keras_accuracy = np.mean(y_pred == y_labels)

print(f"Keras baseline: Accuracy={keras_accuracy:.4f}, FI_Recall={keras_fi_recall:.4f}")

def representative_dataset_gen():
    indices = np.random.choice(len(X_data), size=min(500, len(X_data)), replace=False)
    for i in indices:
        yield [X_data[i:i+1].astype(np.float32)]

def evaluate_tflite(tflite_model, X_test, y_test):
    """Run TFLite inference and evaluate."""
    import time
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_dtype = input_details[0]['dtype']
    
    input_scale = input_details[0].get('quantization_parameters', {}).get('scales', [1.0])
    input_zp = input_details[0].get('quantization_parameters', {}).get('zero_points', [0])
    output_scale = output_details[0].get('quantization_parameters', {}).get('scales', [1.0])
    output_zp = output_details[0].get('quantization_parameters', {}).get('zero_points', [0])
    
    if len(input_scale) > 0: input_scale = input_scale[0]
    if len(input_zp) > 0: input_zp = input_zp[0]
    if len(output_scale) > 0: output_scale = output_scale[0]
    if len(output_zp) > 0: output_zp = output_zp[0]
    
    predictions = []
    inference_times = []
    
    for i in range(len(X_test)):
        sample = X_test[i:i+1].astype(np.float32)
        if input_dtype == np.int8:
            sample = (sample / input_scale + input_zp).astype(np.int8)
        elif input_dtype == np.uint8:
            sample = (sample / input_scale + input_zp).astype(np.uint8)
        
        interpreter.set_tensor(input_details[0]['index'], sample)
        start = time.perf_counter()
        interpreter.invoke()
        elapsed = time.perf_counter() - start
        inference_times.append(elapsed)
        
        output = interpreter.get_tensor(output_details[0]['index'])
        if output_details[0]['dtype'] in [np.int8, np.uint8]:
            output = (output.astype(np.float32) - output_zp) * output_scale
        predictions.append(np.argmax(output, axis=1)[0])
    
    return np.array(predictions), np.array(inference_times)

quant_configs = [
    ('float32',      'none'),
    ('float16',      'float16'),
    ('int8_dynamic', 'int8_weights'),
    ('int8_full',    'int8_full'),
]

quant_results = {}

for name, mode in quant_configs:
    print(f"\n{'─'*60}")
    print(f"Converting: {name}")
    print(f"{'─'*60}")
    
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
        
        if mode == 'float16':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.target_spec.supported_types = [tf.float16]
        elif mode == 'int8_weights':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
        elif mode == 'int8_full':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.representative_dataset = representative_dataset_gen
            converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            converter.inference_input_type = tf.int8
            converter.inference_output_type = tf.int8
        
        tflite_model = converter.convert()
        
        tflite_path = output_dir / f'fallnet_hybrid_staticlmu_{name}.tflite'
        with open(tflite_path, 'wb') as f:
            f.write(tflite_model)
        
        size_kb = len(tflite_model) / 1024
        print(f"  Size: {size_kb:.1f} KB")
        print(f"  Evaluating on {len(X_data):,} samples...")
        
        y_pred_tflite, inf_times = evaluate_tflite(tflite_model, X_data, y_labels)
        
        fi_recall = recall_score(y_labels == fall_init_idx, y_pred_tflite == fall_init_idx)
        fi_f1 = f1_score(y_labels == fall_init_idx, y_pred_tflite == fall_init_idx)
        acc = np.mean(y_pred_tflite == y_labels)
        
        quant_results[name] = {
            'size_kb': size_kb, 'accuracy': acc,
            'fall_init_recall': fi_recall, 'fall_init_f1': fi_f1,
            'mean_inference_us': np.mean(inf_times) * 1e6,
            'tflite_path': str(tflite_path),
        }
        
        print(f"  Accuracy:         {acc:.4f} ({(keras_accuracy - acc)*100:+.2f}%)")
        print(f"  Fall_Init Recall: {fi_recall:.4f} ({(keras_fi_recall - fi_recall)*100:+.2f}%)")
        print(f"  Fall_Init F1:     {fi_f1:.4f}")
        print(f"  ✅ Saved: {tflite_path.name}")
        
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        quant_results[name] = {'error': str(e)}


# %% [markdown]
## 12. Quantization Comparison

# %%
print(f"\n{'='*80}")
print("QUANTIZATION COMPARISON TABLE")
print(f"{'='*80}")

print(f"\n{'Variant':<16s} {'Size (KB)':>10s} {'Accuracy':>10s} {'FI Recall':>10s} {'FI F1':>10s} {'Acc Drop':>10s} {'Compression':>12s}")
print("-" * 82)

valid_results = {k: v for k, v in quant_results.items() if 'error' not in v}
base_size = list(valid_results.values())[0]['size_kb'] if valid_results else 1

for name, r in valid_results.items():
    print(f"{name:<16s} {r['size_kb']:>10.1f} {r['accuracy']:>10.4f} {r['fall_init_recall']:>10.4f} "
          f"{r['fall_init_f1']:>10.4f} {(keras_accuracy - r['accuracy'])*100:>+9.2f}% {base_size/r['size_kb']:>11.1f}x")

# Budget check
print(f"\n{'─'*60}")
print("NANO BLE SENSE REV2 BUDGET CHECK")
print(f"{'─'*60}")
for name, r in valid_results.items():
    flash_pct = r['size_kb'] / 600 * 100
    fit = '✅' if r['size_kb'] < 400 else '⚠️' if r['size_kb'] < 600 else '❌'
    print(f"  {name:<16s}: {r['size_kb']:>7.1f} KB = {flash_pct:>5.1f}% of ~600KB available  {fit}")


# %% [markdown]
## 13. Generate C Header for Deployment

# %%
# Pick best quantized model: smallest that keeps >95% FI recall
best_quant = None
for name in ['int8_full', 'int8_dynamic', 'float16', 'float32']:
    if name in valid_results and valid_results[name]['fall_init_recall'] >= 0.95:
        best_quant = name
        break

if best_quant is None:
    best_quant = list(valid_results.keys())[0]

print(f"\n{'='*80}")
print(f"DEPLOYMENT MODEL: {best_quant.upper()}")
print(f"{'='*80}")

r = valid_results[best_quant]
print(f"Size:             {r['size_kb']:.1f} KB")
print(f"Accuracy:         {r['accuracy']:.4f}")
print(f"Fall_Init Recall: {r['fall_init_recall']:.4f}")

# Generate C header
tflite_path = output_dir / f'fallnet_hybrid_staticlmu_{best_quant}.tflite'
header_path = output_dir / 'fallnet_model.h'

with open(tflite_path, 'rb') as f:
    data = f.read()

with open(header_path, 'w') as f:
    f.write(f'// FallNet CNN→LMU Hybrid (StaticLMU) — {best_quant} quantized\n')
    f.write(f'// Model size: {len(data)} bytes ({len(data)/1024:.1f} KB)\n')
    f.write(f'// Fall_Initiation Recall: {r["fall_init_recall"]:.4f}\n')
    f.write(f'// Overall Accuracy: {r["accuracy"]:.4f}\n')
    f.write(f'// Target: Arduino Nano 33 BLE Sense Rev2 (nRF52840)\n\n')
    f.write(f'#ifndef FALLNET_MODEL_H\n')
    f.write(f'#define FALLNET_MODEL_H\n\n')
    f.write(f'const unsigned int fallnet_model_len = {len(data)};\n')
    f.write(f'alignas(8) const unsigned char fallnet_model[] = {{\n')
    for i in range(0, len(data), 12):
        chunk = data[i:i+12]
        hex_vals = ', '.join(f'0x{b:02x}' for b in chunk)
        f.write(f'  {hex_vals},\n')
    f.write(f'}};\n\n')
    f.write(f'#endif // FALLNET_MODEL_H\n')

print(f"✅ C header saved: {header_path}")
print(f"   Array: fallnet_model[{len(data)}] ({len(data)/1024:.1f} KB)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
names = list(valid_results.keys())
sizes = [valid_results[n]['size_kb'] for n in names]
accs = [valid_results[n]['accuracy'] * 100 for n in names]
fi_recalls = [valid_results[n]['fall_init_recall'] * 100 for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336'][:len(names)]

ax1 = axes[0]
bars = ax1.bar(names, sizes, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Model Size (KB)', fontweight='bold')
ax1.set_title('Model Size by Quantization', fontweight='bold')
ax1.axhline(y=600, color='red', linestyle='--', alpha=0.7, label='Flash budget')
ax1.legend()
ax1_twin = ax1.twinx()
ax1_twin.plot(names, accs, 'ko-', markersize=8, linewidth=2, label='Accuracy %')
ax1_twin.set_ylabel('Accuracy (%)', fontweight='bold')
ax1_twin.legend(loc='center right')

ax2 = axes[1]
ax2.bar(names, fi_recalls, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Fall_Initiation Recall (%)', fontweight='bold')
ax2.set_title('Safety Metric by Quantization', fontweight='bold')
ax2.axhline(y=95, color='red', linestyle='--', alpha=0.7, label='95% threshold')
ax2.axhline(y=99, color='green', linestyle='--', alpha=0.5, label='99% target')
ax2.set_ylim(min(fi_recalls) - 3, 101)
ax2.legend()
for i, r in enumerate(fi_recalls):
    ax2.text(i, r + 0.3, f'{r:.1f}%', ha='center', fontweight='bold')

plt.suptitle('FallNet CNN→LMU (StaticLMU) — Quantization Tradeoffs', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / 'quantization_comparison_staticlmu.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Quantization comparison chart saved")


# %% [markdown]
## 14. Final Summary

# %%
stats = fallnet.get_model_stats()

print(f"\n{'='*80}")
print("TRAINING + QUANTIZATION COMPLETE — FINAL SUMMARY")
print(f"{'='*80}")

summary = f"""
✅ FallNet {MODEL_VARIANT.upper()} with StaticLMU — 5-fold CV + Quantization

Architecture: CNN→StaticLMU Hybrid (TFLite-compatible, static unrolling)
  - StaticLMU replaces keras_lmu.LMU (eliminates TensorListReserve ops)
  - Identical LMU math (Legendre memory), different execution strategy

Model: {stats['total']:,} params

Average Performance (5-fold CV):
  - Accuracy:  {mean_results['val_accuracy']:.4f} ± {std_results['val_accuracy']:.4f}
  - Precision: {mean_results['val_precision']:.4f} ± {std_results['val_precision']:.4f}
  - Recall:    {mean_results['val_recall']:.4f} ± {std_results['val_recall']:.4f}
  - F1-Score:  {mean_results['val_f1']:.4f} ± {std_results['val_f1']:.4f}

Fall_Initiation (Keras): Recall={keras_fi_recall:.4f}, F1={keras_fi_f1:.4f}

Quantization Results:
"""

for name, r in valid_results.items():
    summary += f"  {name:<16s}: {r['size_kb']:>7.1f} KB | Acc: {r['accuracy']:.4f} | FI Recall: {r['fall_init_recall']:.4f}\n"

summary += f"""
Deployment Model: {best_quant}
  Size: {valid_results[best_quant]['size_kb']:.1f} KB
  Accuracy: {valid_results[best_quant]['accuracy']:.4f}
  FI Recall: {valid_results[best_quant]['fall_init_recall']:.4f}

Target: Arduino Nano 33 BLE Sense Rev2 (nRF52840, 1MB flash, 256KB RAM)
Files: fallnet_hybrid_staticlmu_{best_quant}.tflite, fallnet_model.h
"""

print(summary)
with open(output_dir / f'training_summary_{MODEL_VARIANT}_staticlmu.txt', 'w') as f:
    f.write(summary)
print(f"✅ Summary saved")
print(f"\n🎯 Next: Flash fallnet_model.h to Nano BLE Sense, measure power with PPK2")

In [ ]:
# %% [markdown]
# # FallNet with Static-Unrolled LMU — TFLite Compatible
# 
# Replaces keras_lmu.LMU with a hand-written layer that statically unrolls
# the LMU recurrence. This eliminates tf.TensorListReserve / tf.while_loop
# ops that block TFLite conversion.
#
# The LMU math is identical — only the execution strategy changes:
#   keras_lmu: RNN(LMUCell) → dynamic tf.while_loop → TensorArrays → TFLite fails
#   StaticLMU: explicit Python for-loop at graph build time → static ops → TFLite works
#
# Drop-in replacement: just swap `from keras_lmu import LMU` for this file's StaticLMU.

# %%
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")


# %% [markdown]
## 1. Static LMU Layer (TFLite-Compatible)

# %%
@keras.utils.register_keras_serializable()
class StaticLMUCell(layers.Layer):
    """
    Single-step LMU cell for manual unrolling.
    
    Implements the Legendre Memory Unit recurrence:
        m[t] = A @ m[t-1] + B @ x[t]     (memory state update via Legendre ODE)
        h[t] = hidden_cell([x[t]; m[t]])   (hidden state via LSTM/Dense)
    
    where A, B are the discretized Legendre matrices (fixed, not learned).
    
    This cell is called once per timestep in a Python for-loop,
    producing a static TF graph with no tf.while_loop or TensorArrays.
    """
    
    def __init__(self, input_dim, memory_d, order, theta, hidden_units, 
                 dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.input_dim = input_dim
        self.memory_d = memory_d
        self.order = order
        self.theta = theta
        self.hidden_units = hidden_units
        self.dropout_rate = dropout
        
        # Memory state size: memory_d dimensions * order polynomials
        self.memory_size = memory_d * order
        
    def build(self, input_shape):
        # Compute the discretized Legendre matrices A, B
        # These are FIXED (non-trainable) — derived from the Legendre ODE
        A, B = self._get_legendre_matrices(self.order, self.theta)
        
        # Store as non-trainable weights so they're saved with the model
        self.A = self.add_weight(
            name='A', shape=(self.order, self.order),
            initializer=keras.initializers.Constant(A),
            trainable=False
        )
        self.B = self.add_weight(
            name='B', shape=(self.order, 1),
            initializer=keras.initializers.Constant(B),
            trainable=False
        )
        
        # Encoder: project input features to memory_d dimensions
        # Each of the memory_d dimensions gets its own projection of the input
        self.encoder = layers.Dense(
            self.memory_d,
            kernel_regularizer=keras.regularizers.l2(1e-5),
            name='lmu_encoder'
        )
        
        # Hidden cell: processes [input; flattened_memory] → hidden_state
        # Using Dense + tanh instead of LSTMCell for TFLite compatibility
        # (LSTMCell also uses internal state management that can cause issues)
        self.hidden_dense1 = layers.Dense(
            self.hidden_units,
            activation='tanh',
            kernel_regularizer=keras.regularizers.l2(1e-5),
            name='lmu_hidden1'
        )
        self.hidden_dense2 = layers.Dense(
            self.hidden_units,
            activation='tanh',
            kernel_regularizer=keras.regularizers.l2(1e-5),
            name='lmu_hidden2'
        )
        
        if self.dropout_rate > 0:
            self.dropout = layers.Dropout(self.dropout_rate)
        
        super().build(input_shape)
    
    @staticmethod
    def _get_legendre_matrices(order, theta):
        """
        Compute the discretized Legendre delay matrices A and B.
        
        These encode the continuous-time Legendre ODE (HiPPO-LegS):
            θ * dm/dt = A_cont @ m + B_cont @ x
        
        Discretized via ZOH (zero-order hold) using matrix exponential:
            A_d = expm(A_cont / θ)
            B_d = A_cont^{-1} @ (A_d - I) @ B_cont
        
        This guarantees stability (all eigenvalues of A_d inside unit circle)
        unlike Euler discretization which diverges for high order/low theta.
        
        For order=64, theta=50:
            Euler:  max|eig| = 3.13 → UNSTABLE (NaN after ~10 steps)
            ZOH:    max|eig| = 0.80 → stable
        """
        from scipy.linalg import expm
        
        Q = np.arange(order, dtype=np.float64)
        R = (2 * Q + 1)[:, None]
        j, i = np.meshgrid(Q, Q)
        
        # Continuous-time matrices
        A_cont = np.where(i < j, -1, (-1.0) ** (i - j + 1)) * R
        B_cont = (-1.0) ** Q[:, None] * R
        
        # ZOH discretization via matrix exponential
        A_d = expm(A_cont / theta)
        
        # B_d = A_cont^{-1} @ (A_d - I) @ B_cont
        B_d = np.linalg.solve(A_cont, (A_d - np.eye(order)) @ B_cont)
        
        return A_d.astype(np.float32), B_d.astype(np.float32)
    
    def call(self, x_t, memory_state, training=False):
        """
        Single timestep update.
        
        Args:
            x_t: input at time t, shape (batch, input_dim)
            memory_state: previous memory, shape (batch, memory_d, order)
            training: whether in training mode (for dropout)
            
        Returns:
            h_t: hidden output, shape (batch, hidden_units)
            new_memory: updated memory, shape (batch, memory_d, order)
        """
        # Encode input to memory_d dimensions: (batch, input_dim) → (batch, memory_d)
        u_t = self.encoder(x_t)  # (batch, memory_d)
        
        # Update memory state for each memory dimension
        # m[t] = A @ m[t-1] + B @ u[t]
        # memory_state: (batch, memory_d, order)
        # A: (order, order), B: (order, 1)
        
        # memory_state: (batch, memory_d, order)
        # A: (order, order)
        # For each memory dim, we need: new_m_d = m_d @ A^T  (vector-matrix product along order dim)
        # Batched: (batch, memory_d, order) @ (order, order) = (batch, memory_d, order)
        Am = tf.matmul(memory_state, self.A, transpose_b=True)  # (batch, memory_d, order)
        
        # B @ u_t: B is (order, 1), u_t is (batch, memory_d)
        # We want (batch, memory_d, order): each memory dim scaled by its u_t value
        # B squeezed: (order,) → broadcast with u_t: (batch, memory_d, 1) * (1, order)
        Bu = tf.expand_dims(u_t, axis=-1) * tf.reshape(self.B, [1, 1, self.order])
        # Bu shape: (batch, memory_d, order)
        
        new_memory = Am + Bu  # (batch, memory_d, order)
        
        # Flatten memory for hidden cell input
        m_flat = tf.reshape(new_memory, [-1, self.memory_size])  # (batch, memory_d * order)
        
        # Concatenate input with flattened memory
        h_input = tf.concat([x_t, m_flat], axis=-1)  # (batch, input_dim + memory_d * order)
        
        # Process through hidden layers
        h_t = self.hidden_dense1(h_input)
        if self.dropout_rate > 0:
            h_t = self.dropout(h_t, training=training)
        h_t = self.hidden_dense2(h_t)
        
        return h_t, new_memory


@keras.saving.register_keras_serializable()
class StaticLMU(layers.Layer):
    """
    TFLite-compatible LMU layer with static unrolling.
    
    Instead of using tf.keras.layers.RNN (which creates dynamic TensorArrays),
    this layer unrolls the LMU recurrence in a Python for-loop at graph 
    construction time. The resulting TF graph has only static ops.
    
    The sequence length MUST be known at build time (which it is for our
    CNN→LMU hybrid: CNN always outputs exactly 50 timesteps).
    
    Usage:
        # Replace:
        #   from keras_lmu import LMU
        #   x = LMU(memory_d=4, order=64, theta=50, ...)(inputs)
        # With:
        x = StaticLMU(memory_d=4, order=64, theta=50, hidden_units=128)(inputs)
    """
    
    def __init__(self, memory_d, order, theta, hidden_units, dropout=0.0, 
                 return_sequences=False, **kwargs):
        super().__init__(**kwargs)
        self.memory_d = memory_d
        self.order = order
        self.theta = theta
        self.hidden_units = hidden_units
        self.dropout_rate = dropout
        self.return_sequences = return_sequences
        
    def build(self, input_shape):
        # input_shape: (batch, timesteps, features)
        self.timesteps = input_shape[1]
        self.input_dim = input_shape[2]
        
        if self.timesteps is None:
            raise ValueError(
                "StaticLMU requires a known sequence length at build time. "
                "Got input shape with timesteps=None. "
                "Ensure the input tensor has a static time dimension."
            )
        
        self.cell = StaticLMUCell(
            input_dim=self.input_dim,
            memory_d=self.memory_d,
            order=self.order,
            theta=self.theta,
            hidden_units=self.hidden_units,
            dropout=self.dropout_rate,
            name='lmu_cell'
        )
        
        super().build(input_shape)
        
    def call(self, inputs, training=False):
        """
        Forward pass with static unrolling.
        
        Args:
            inputs: (batch, timesteps, features)
            training: whether in training mode
            
        Returns:
            If return_sequences=False: last hidden state (batch, hidden_units)
            If return_sequences=True: all hidden states (batch, timesteps, hidden_units)
        """
        batch_size = tf.shape(inputs)[0]
        
        # Initialize memory state to zeros
        memory = tf.zeros([batch_size, self.memory_d, self.order])
        
        # Split input along time axis — this creates self.timesteps separate tensors
        # Each is (batch, features), and the list has a known Python length
        x_steps = tf.unstack(inputs, num=self.timesteps, axis=1)
        
        if self.return_sequences:
            outputs = []
        
        # Static unroll: Python for-loop generates fixed graph ops
        # No tf.while_loop, no TensorArrays, no dynamic shapes
        for t in range(self.timesteps):
            h_t, memory = self.cell(x_steps[t], memory, training=training)
            if self.return_sequences:
                outputs.append(h_t)
        
        if self.return_sequences:
            return tf.stack(outputs, axis=1)  # (batch, timesteps, hidden_units)
        else:
            return h_t  # (batch, hidden_units) — last timestep only
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'memory_d': self.memory_d,
            'order': self.order,
            'theta': self.theta,
            'hidden_units': self.hidden_units,
            'dropout': self.dropout_rate,
            'return_sequences': self.return_sequences,
        })
        return config


print("✅ StaticLMU layer defined (TFLite-compatible, static unrolling)")


# %% [markdown]
## 2. FallNet with StaticLMU

# %%
class FallNet:
    """
    FallNet: CNN-LMU Architectures for Pre-Impact Fall Detection
    
    Now using StaticLMU instead of keras_lmu.LMU for TFLite compatibility.
    
    Supports 4 model variants for ablation study:
      1. CNN-only         — spatial features only (embedded baseline)
      2. LMU-only         — temporal modeling on raw input (StaticLMU)
      3. CNN→LMU Hybrid   — CNN spatial extraction → StaticLMU temporal backbone
      4. CNN+LMU Ensemble — parallel branches with averaged outputs
    """
    
    def __init__(self, input_shape=(200, 6), n_classes=6):
        self.input_shape = input_shape
        self.n_classes = n_classes
        self.model = None
    
    # =========================================================================
    # Building Blocks
    # =========================================================================
    
    def _cnn_feature_extractor(self, inputs, name_prefix='cnn'):
        """
        CNN spatial feature extractor.
        Input: (batch, 200, 6) → Output: (batch, 50, 64)
        """
        x = layers.Conv1D(
            filters=32, kernel_size=5, activation='relu', padding='same',
            kernel_regularizer=keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv1'
        )(inputs)
        x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
        x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_pool1')(x)
        
        x = layers.Conv1D(
            filters=64, kernel_size=3, activation='relu', padding='same',
            kernel_regularizer=keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv2'
        )(x)
        x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
        x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_pool2')(x)
        x = layers.Dropout(0.2, name=f'{name_prefix}_drop1')(x)
        
        return x
    
    def _lmu_temporal_block(self, inputs, memory_d=4, order=64, theta=100.0,
                            hidden_units=128, name_prefix='lmu'):
        """
        StaticLMU temporal modeling block (TFLite-compatible).
        Uses static unrolling — no dynamic ops.
        """
        x = StaticLMU(
            memory_d=memory_d,
            order=order,
            theta=theta,
            hidden_units=hidden_units,
            dropout=0.2,
            return_sequences=False,
            name=f'{name_prefix}_layer'
        )(inputs)
        
        return x
    
    def _classification_head(self, features, hidden_dims=[128, 64],
                             dropout_rate=0.3, name_prefix='head'):
        """Shared classification head."""
        x = features
        for i, dim in enumerate(hidden_dims):
            x = layers.Dense(dim, activation='relu', name=f'{name_prefix}_dense{i+1}')(x)
            x = layers.BatchNormalization(name=f'{name_prefix}_bn{i+1}')(x)
            x = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop{i+1}')(x)
        
        output = layers.Dense(
            self.n_classes, activation='softmax', name=f'{name_prefix}_output'
        )(x)
        return output
    
    # =========================================================================
    # Model Variants
    # =========================================================================
    
    def build_cnn_only(self):
        """CNN-only model — spatial features with global pooling."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        x = layers.GlobalAveragePooling1D(name='global_pool')(cnn_features)
        output = self._classification_head(x, hidden_dims=[128, 64], name_prefix='cnn_head')
        self.model = models.Model(inputs=inputs, outputs=output, name='FallNet_CNN_Only')
        return self.model
    
    def build_lmu_only(self):
        """LMU-only model — temporal modeling directly on raw IMU signals."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        lmu_features = self._lmu_temporal_block(
            inputs, memory_d=2, order=64, theta=200.0, hidden_units=128,
            name_prefix='lmu'
        )
        output = self._classification_head(lmu_features, hidden_dims=[512, 128],
                                           name_prefix='lmu_head')
        self.model = models.Model(inputs=inputs, outputs=output, name='FallNet_LMU_Only')
        return self.model
    
    def build_cnn_lmu_hybrid(self):
        """
        CNN→LMU Hybrid — the main contribution.
        CNN extracts spatial features, StaticLMU models temporal dynamics.
        theta=50 because CNN compresses 200→50 timesteps.
        """
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        
        lmu_features = self._lmu_temporal_block(
            cnn_features,
            memory_d=4, order=64, theta=50.0, hidden_units=128,
            name_prefix='lmu'
        )
        
        output = self._classification_head(lmu_features, hidden_dims=[128, 64],
                                           name_prefix='hybrid_head')
        
        self.model = models.Model(inputs=inputs, outputs=output,
                                  name='FallNet_CNN_LMU_Hybrid')
        return self.model
    
    def build_ensemble(self):
        """CNN+LMU Ensemble — parallel branches, averaged outputs."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        cnn_features = self._cnn_feature_extractor(inputs, name_prefix='cnn')
        cnn_pooled = layers.GlobalAveragePooling1D(name='cnn_global_pool')(cnn_features)
        cnn_output = self._classification_head(cnn_pooled, hidden_dims=[128, 64],
                                               name_prefix='cnn_head')
        
        lmu_features = self._lmu_temporal_block(
            inputs, memory_d=2, order=64, theta=200.0, hidden_units=128,
            name_prefix='lmu'
        )
        lmu_output = self._classification_head(lmu_features, hidden_dims=[512, 128],
                                               name_prefix='lmu_head')
        
        ensemble_output = layers.Average(name='ensemble_average')([cnn_output, lmu_output])
        self.model = models.Model(inputs=inputs, outputs=ensemble_output,
                                  name='FallNet_CNN_LMU_Ensemble')
        return self.model
    
    # =========================================================================
    # Compilation & Stats
    # =========================================================================
    
    def compile_model(self, learning_rate=5e-4):
        if self.model is None:
            raise ValueError("Model not built yet.")
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy', keras.metrics.Precision(name='precision'),
                     keras.metrics.Recall(name='recall')]
        )
        return self.model
    
    def get_model_stats(self):
        if self.model is None:
            raise ValueError("No model built yet.")
        trainable = np.sum([np.prod(v.shape) for v in self.model.trainable_weights])
        non_trainable = np.sum([np.prod(v.shape) for v in self.model.non_trainable_weights])
        total = trainable + non_trainable
        float32_size_mb = total * 4 / (1024 * 1024)
        int8_size_kb = total * 1 / 1024
        
        print(f"\n{'='*60}")
        print(f"MODEL: {self.model.name}")
        print(f"{'='*60}")
        print(f"Trainable params:     {trainable:>10,}")
        print(f"Non-trainable params: {non_trainable:>10,}")
        print(f"Total params:         {total:>10,}")
        print(f"Float32 size:         {float32_size_mb:>10.2f} MB")
        print(f"INT8 quantized (est): {int8_size_kb:>10.1f} KB")
        print(f"Nano BLE flash (1MB): {'✅ fits' if int8_size_kb < 500 else '⚠️  tight' if int8_size_kb < 900 else '❌ too large'}")
        print(f"{'='*60}")
        
        return {'trainable': trainable, 'non_trainable': non_trainable,
                'total': total, 'float32_mb': float32_size_mb, 'int8_kb': int8_size_kb}

print("✅ FallNet class defined with StaticLMU (4 variants, TFLite-compatible)")


# %% [markdown]
## 3. Verify TFLite Conversion BEFORE Training
# Quick sanity check that the architecture actually converts.

# %%
print("\n" + "="*80)
print("TFLITE CONVERSION SMOKE TEST")
print("="*80)

with tf.device('/CPU:0'):
    test_fn = FallNet(input_shape=(200, 6), n_classes=6)
    test_model = test_fn.build_cnn_lmu_hybrid()
    test_fn.compile_model()
    test_fn.get_model_stats()

# Try converting
print("\nAttempting TFLite conversion...")
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(test_model)
    tflite_bytes = converter.convert()
    print(f"✅ Float32 TFLite conversion succeeded: {len(tflite_bytes)/1024:.1f} KB")
    
    # Try INT8 quantization too
    converter2 = tf.lite.TFLiteConverter.from_keras_model(test_model)
    converter2.optimizations = [tf.lite.Optimize.DEFAULT]
    
    def rep_data():
        for _ in range(50):
            yield [np.random.randn(1, 200, 6).astype(np.float32)]
    
    converter2.representative_dataset = rep_data
    converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter2.inference_input_type = tf.int8
    converter2.inference_output_type = tf.int8
    tflite_int8 = converter2.convert()
    print(f"✅ INT8 TFLite conversion succeeded: {len(tflite_int8)/1024:.1f} KB")
    
    print("\n🎯 Both quantization levels work — safe to train!")
    
except Exception as e:
    print(f"❌ TFLite conversion failed: {e}")
    print("\nDo NOT proceed with training — fix the layer first.")

del test_fn, test_model
print()


# %% [markdown]
## 4. Build and Compare All Architectures

# %%
print("\n" + "="*80)
print("ARCHITECTURE COMPARISON (StaticLMU)")
print("="*80)

model_stats = {}
for variant_name, build_fn in [
    ('CNN-only',       'build_cnn_only'),
    ('LMU-only',       'build_lmu_only'),
    ('CNN→LMU Hybrid', 'build_cnn_lmu_hybrid'),
    ('Ensemble',       'build_ensemble'),
]:
    with tf.device('/CPU:0'):
        fn = FallNet(input_shape=(200, 6), n_classes=6)
        getattr(fn, build_fn)()
        stats = fn.get_model_stats()
        model_stats[variant_name] = stats

print("\n" + "="*80)
print("PARAMETER COMPARISON SUMMARY")
print("="*80)
print(f"\n{'Model':<20s} {'Total Params':>14s} {'Float32 (MB)':>14s} {'INT8 (KB)':>12s} {'Nano Fit?':>10s}")
print("-" * 74)
for name, s in model_stats.items():
    fit = '✅' if s['int8_kb'] < 500 else '⚠️' if s['int8_kb'] < 900 else '❌'
    print(f"{name:<20s} {s['total']:>14,} {s['float32_mb']:>14.2f} {s['int8_kb']:>12.1f} {fit:>10s}")


# %% [markdown]
## 5. Select Model & Train

# %%
MODEL_VARIANT = 'hybrid'  # Options: 'cnn_only', 'lmu_only', 'hybrid', 'ensemble'

VARIANT_MAP = {
    'cnn_only':  'build_cnn_only',
    'lmu_only':  'build_lmu_only',
    'hybrid':    'build_cnn_lmu_hybrid',
    'ensemble':  'build_ensemble',
}

print(f"\n{'='*80}")
print(f"SELECTED MODEL: {MODEL_VARIANT.upper()}")
print(f"{'='*80}")

with tf.device('/CPU:0'):
    fallnet = FallNet(input_shape=(200, 6), n_classes=6)
    model = getattr(fallnet, VARIANT_MAP[MODEL_VARIANT])()

model = fallnet.compile_model()
model.summary()
fallnet.get_model_stats()


# %% [markdown]
## 6. Training Configuration

# %%
BATCH_SIZE = 128
EPOCHS = 50
K_FOLDS = 5

print(f"\n{'='*80}")
print("TRAINING CONFIGURATION")
print(f"{'='*80}")
print(f"Model:      {MODEL_VARIANT}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {EPOCHS}")
print(f"K-Folds:    {K_FOLDS}")
print(f"Using data: {len(y_labels):,} samples, {len(np.unique(y_labels))} classes")


# %% [markdown]
## 7. Pre-Training Verification

# %%
print(f"\n{'='*80}")
print("PRE-TRAINING VERIFICATION")
print(f"{'='*80}")
print(f"✅ Data shapes:")
print(f"   X_data:        {X_data.shape}")
print(f"   y_labels:      {y_labels.shape}")
print(f"   y_categorical: {y_categorical.shape}")
print(f"\n✅ Classes: {len(np.unique(y_labels))} (should be 6)")
print(f"✅ Label range: {y_labels.min()}-{y_labels.max()} (should be 0-5)")
print(f"✅ Model output: {model.output_shape[-1]} (should be 6)")

assert X_data.shape[0] == y_labels.shape[0] == y_categorical.shape[0], "Shape mismatch!"
assert len(np.unique(y_labels)) == 6, "Should have 6 classes!"
assert y_labels.max() == 5, "Max label should be 5!"
assert model.output_shape[-1] == 6, "Model should output 6 classes!"

print("\n✅ All checks passed — ready to train!")


# %% [markdown]
## 8. K-Fold Cross-Validation Training

# %%
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_results = []
fold_histories = []

print(f"\n{'='*80}")
print(f"STARTING K-FOLD CROSS-VALIDATION — {MODEL_VARIANT.upper()} (StaticLMU)")
print(f"{'='*80}")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"{'='*80}")
    
    X_train, X_val = X_data[train_idx], X_data[val_idx]
    y_train, y_val = y_categorical[train_idx], y_categorical[val_idx]
    y_train_labels = y_labels[train_idx]
    
    print(f"Train: {X_train.shape[0]:,} samples | Val: {X_val.shape[0]:,} samples")
    
    # Build fresh model
    fallnet_fold = FallNet(input_shape=(200, 6), n_classes=6)
    model_fold = getattr(fallnet_fold, VARIANT_MAP[MODEL_VARIANT])()
    model_fold = fallnet_fold.compile_model()
    
    fold_callbacks = [
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1),
        ModelCheckpoint(
            filepath=str(output_dir / f'fallnet_{MODEL_VARIANT}_staticlmu_fold_{fold}.keras'),
            monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
        )
    ]
    
    # Class weights
    class_weights_array = compute_class_weight(
        class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels
    )
    class_weights = dict(enumerate(class_weights_array))
    class_weights[0] *= 1.5   # Walking
    class_weights[3] *= 3.0   # Stumbles
    class_weights[4] *= 1.2   # Fall Initiation
    MAX_WEIGHT = 5.0
    for k in class_weights:
        class_weights[k] = min(class_weights[k], MAX_WEIGHT)
    
    if fold == 1:
        print("\nClass Weights:")
        for cls_idx in range(6):
            print(f"  {reverse_label_map[cls_idx]:<30s}: {class_weights[cls_idx]:.2f}x")
    
    print(f"\nTraining fold {fold}...")
    history = model_fold.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=fold_callbacks,
        verbose=1
    )
    
    val_loss, val_acc, val_precision, val_recall = model_fold.evaluate(
        X_val, y_val, batch_size=2, verbose=0
    )
    val_f1 = (2 * (val_precision * val_recall) / (val_precision + val_recall)
              if (val_precision + val_recall) > 0 else 0)
    
    print(f"\n{'='*50}")
    print(f"Fold {fold} Results:")
    print(f"{'='*50}")
    print(f"Loss:      {val_loss:.4f}")
    print(f"Accuracy:  {val_acc:.4f}")
    print(f"Precision: {val_precision:.4f}")
    print(f"Recall:    {val_recall:.4f}")
    print(f"F1-Score:  {val_f1:.4f}")
    
    fold_results.append({
        'fold': fold, 'val_loss': val_loss, 'val_accuracy': val_acc,
        'val_precision': val_precision, 'val_recall': val_recall, 'val_f1': val_f1
    })
    fold_histories.append(history.history)
    print(f"✅ Model saved: fallnet_{MODEL_VARIANT}_staticlmu_fold_{fold}.keras")

print(f"\n{'='*80}")
print("K-FOLD CROSS-VALIDATION COMPLETE")
print(f"{'='*80}")


# %% [markdown]
## 9. Results & Evaluation (same as before)

# %%
results_df = pd.DataFrame(fold_results)

print(f"\n{'='*80}")
print("RESULTS ACROSS ALL FOLDS")
print(f"{'='*80}")
print(results_df.to_string(index=False))

mean_results = results_df.mean(numeric_only=True)
std_results = results_df.std(numeric_only=True)

print(f"\n{'='*80}")
print("AVERAGE PERFORMANCE ± STD")
print(f"{'='*80}")
for metric in ['val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1']:
    print(f"  {metric:<16s}: {mean_results[metric]:.4f} ±{std_results[metric]:.4f}")

# Training history visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
metrics = [('loss', 'Loss'), ('accuracy', 'Accuracy'), ('precision', 'Precision'), ('recall', 'Recall')]
for idx, (metric, title) in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    for fold_num, history in enumerate(fold_histories, 1):
        epochs = range(1, len(history[metric]) + 1)
        ax.plot(epochs, history[metric], label=f'Fold {fold_num} Train', alpha=0.5, linewidth=1)
        ax.plot(epochs, history[f'val_{metric}'], label=f'Fold {fold_num} Val',
                linestyle='--', alpha=0.7, linewidth=1.5)
    ax.set_title(f'{title} Across All Folds', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle(f'FallNet {MODEL_VARIANT.upper()} (StaticLMU) — 5-Fold CV',
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(output_dir / f'training_history_{MODEL_VARIANT}_staticlmu.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training history saved")


# %% [markdown]
## 10. Best Fold Evaluation

# %%
best_fold = int(results_df.loc[results_df['val_f1'].idxmax(), 'fold'])
print(f"\n{'='*80}")
print(f"DETAILED EVALUATION — BEST FOLD #{best_fold}")
print(f"{'='*80}")

best_model = keras.models.load_model(
    output_dir / f'fallnet_{MODEL_VARIANT}_staticlmu_fold_{best_fold}.keras'
)

y_pred_probs = best_model.predict(X_data, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
class_names = [reverse_label_map[i] for i in range(6)]

print(f"\n{'='*80}")
print(f"CLASSIFICATION REPORT — {MODEL_VARIANT.upper()} StaticLMU")
print(f"{'='*80}")
print(classification_report(y_labels, y_pred, target_names=class_names, digits=4))

# Per-class metrics
print(f"\n{'Class':<40s} {'Precision':<12s} {'Recall':<12s} {'F1-Score':<12s} {'Support'}")
print("-" * 90)
for cls_idx in range(6):
    p = precision_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    r = recall_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    f = f1_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    s = np.sum(y_labels == cls_idx)
    print(f"{reverse_label_map[cls_idx]:<40s} {p:<12.4f} {r:<12.4f} {f:<12.4f} {s}")

# Confusion matrix
cm = confusion_matrix(y_labels, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix — {MODEL_VARIANT.upper()} StaticLMU', fontsize=15, fontweight='bold', pad=20)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(output_dir / f'confusion_matrix_{MODEL_VARIANT}_staticlmu.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix saved")


# %% [markdown]
## 11. TFLite Quantization — All Variants

# %%
print(f"\n{'='*80}")
print("TFLITE QUANTIZATION — TRAINED MODEL")
print(f"{'='*80}")

# Fall_Initiation metrics
fall_init_idx = label_map["Fall_Initiation"]
keras_fi_recall = recall_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
keras_fi_f1 = f1_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
keras_accuracy = np.mean(y_pred == y_labels)

print(f"Keras baseline: Accuracy={keras_accuracy:.4f}, FI_Recall={keras_fi_recall:.4f}")

def representative_dataset_gen():
    indices = np.random.choice(len(X_data), size=min(500, len(X_data)), replace=False)
    for i in indices:
        yield [X_data[i:i+1].astype(np.float32)]

def evaluate_tflite(tflite_model, X_test, y_test):
    """Run TFLite inference and evaluate."""
    import time
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_dtype = input_details[0]['dtype']
    
    input_scale = input_details[0].get('quantization_parameters', {}).get('scales', [1.0])
    input_zp = input_details[0].get('quantization_parameters', {}).get('zero_points', [0])
    output_scale = output_details[0].get('quantization_parameters', {}).get('scales', [1.0])
    output_zp = output_details[0].get('quantization_parameters', {}).get('zero_points', [0])
    
    if len(input_scale) > 0: input_scale = input_scale[0]
    if len(input_zp) > 0: input_zp = input_zp[0]
    if len(output_scale) > 0: output_scale = output_scale[0]
    if len(output_zp) > 0: output_zp = output_zp[0]
    
    predictions = []
    inference_times = []
    
    for i in range(len(X_test)):
        sample = X_test[i:i+1].astype(np.float32)
        if input_dtype == np.int8:
            sample = (sample / input_scale + input_zp).astype(np.int8)
        elif input_dtype == np.uint8:
            sample = (sample / input_scale + input_zp).astype(np.uint8)
        
        interpreter.set_tensor(input_details[0]['index'], sample)
        start = time.perf_counter()
        interpreter.invoke()
        elapsed = time.perf_counter() - start
        inference_times.append(elapsed)
        
        output = interpreter.get_tensor(output_details[0]['index'])
        if output_details[0]['dtype'] in [np.int8, np.uint8]:
            output = (output.astype(np.float32) - output_zp) * output_scale
        predictions.append(np.argmax(output, axis=1)[0])
    
    return np.array(predictions), np.array(inference_times)

quant_configs = [
    ('float32',      'none'),
    ('float16',      'float16'),
    ('int8_dynamic', 'int8_weights'),
    ('int8_full',    'int8_full'),
]

quant_results = {}

for name, mode in quant_configs:
    print(f"\n{'─'*60}")
    print(f"Converting: {name}")
    print(f"{'─'*60}")
    
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
        
        if mode == 'float16':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.target_spec.supported_types = [tf.float16]
        elif mode == 'int8_weights':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
        elif mode == 'int8_full':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.representative_dataset = representative_dataset_gen
            converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            converter.inference_input_type = tf.int8
            converter.inference_output_type = tf.int8
        
        tflite_model = converter.convert()
        
        tflite_path = output_dir / f'fallnet_hybrid_staticlmu_{name}.tflite'
        with open(tflite_path, 'wb') as f:
            f.write(tflite_model)
        
        size_kb = len(tflite_model) / 1024
        print(f"  Size: {size_kb:.1f} KB")
        print(f"  Evaluating on {len(X_data):,} samples...")
        
        y_pred_tflite, inf_times = evaluate_tflite(tflite_model, X_data, y_labels)
        
        fi_recall = recall_score(y_labels == fall_init_idx, y_pred_tflite == fall_init_idx)
        fi_f1 = f1_score(y_labels == fall_init_idx, y_pred_tflite == fall_init_idx)
        acc = np.mean(y_pred_tflite == y_labels)
        
        quant_results[name] = {
            'size_kb': size_kb, 'accuracy': acc,
            'fall_init_recall': fi_recall, 'fall_init_f1': fi_f1,
            'mean_inference_us': np.mean(inf_times) * 1e6,
            'tflite_path': str(tflite_path),
        }
        
        print(f"  Accuracy:         {acc:.4f} ({(keras_accuracy - acc)*100:+.2f}%)")
        print(f"  Fall_Init Recall: {fi_recall:.4f} ({(keras_fi_recall - fi_recall)*100:+.2f}%)")
        print(f"  Fall_Init F1:     {fi_f1:.4f}")
        print(f"  ✅ Saved: {tflite_path.name}")
        
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        quant_results[name] = {'error': str(e)}


# %% [markdown]
## 12. Quantization Comparison

# %%
print(f"\n{'='*80}")
print("QUANTIZATION COMPARISON TABLE")
print(f"{'='*80}")

print(f"\n{'Variant':<16s} {'Size (KB)':>10s} {'Accuracy':>10s} {'FI Recall':>10s} {'FI F1':>10s} {'Acc Drop':>10s} {'Compression':>12s}")
print("-" * 82)

valid_results = {k: v for k, v in quant_results.items() if 'error' not in v}
base_size = list(valid_results.values())[0]['size_kb'] if valid_results else 1

for name, r in valid_results.items():
    print(f"{name:<16s} {r['size_kb']:>10.1f} {r['accuracy']:>10.4f} {r['fall_init_recall']:>10.4f} "
          f"{r['fall_init_f1']:>10.4f} {(keras_accuracy - r['accuracy'])*100:>+9.2f}% {base_size/r['size_kb']:>11.1f}x")

# Budget check
print(f"\n{'─'*60}")
print("NANO BLE SENSE REV2 BUDGET CHECK")
print(f"{'─'*60}")
for name, r in valid_results.items():
    flash_pct = r['size_kb'] / 600 * 100
    fit = '✅' if r['size_kb'] < 400 else '⚠️' if r['size_kb'] < 600 else '❌'
    print(f"  {name:<16s}: {r['size_kb']:>7.1f} KB = {flash_pct:>5.1f}% of ~600KB available  {fit}")


# %% [markdown]
## 13. Generate C Header for Deployment

# %%
# Pick best quantized model: smallest that keeps >95% FI recall
best_quant = None
for name in ['int8_full', 'int8_dynamic', 'float16', 'float32']:
    if name in valid_results and valid_results[name]['fall_init_recall'] >= 0.95:
        best_quant = name
        break

if best_quant is None:
    best_quant = list(valid_results.keys())[0]

print(f"\n{'='*80}")
print(f"DEPLOYMENT MODEL: {best_quant.upper()}")
print(f"{'='*80}")

r = valid_results[best_quant]
print(f"Size:             {r['size_kb']:.1f} KB")
print(f"Accuracy:         {r['accuracy']:.4f}")
print(f"Fall_Init Recall: {r['fall_init_recall']:.4f}")

# Generate C header
tflite_path = output_dir / f'fallnet_hybrid_staticlmu_{best_quant}.tflite'
header_path = output_dir / 'fallnet_model.h'

with open(tflite_path, 'rb') as f:
    data = f.read()

with open(header_path, 'w') as f:
    f.write(f'// FallNet CNN→LMU Hybrid (StaticLMU) — {best_quant} quantized\n')
    f.write(f'// Model size: {len(data)} bytes ({len(data)/1024:.1f} KB)\n')
    f.write(f'// Fall_Initiation Recall: {r["fall_init_recall"]:.4f}\n')
    f.write(f'// Overall Accuracy: {r["accuracy"]:.4f}\n')
    f.write(f'// Target: Arduino Nano 33 BLE Sense Rev2 (nRF52840)\n\n')
    f.write(f'#ifndef FALLNET_MODEL_H\n')
    f.write(f'#define FALLNET_MODEL_H\n\n')
    f.write(f'const unsigned int fallnet_model_len = {len(data)};\n')
    f.write(f'alignas(8) const unsigned char fallnet_model[] = {{\n')
    for i in range(0, len(data), 12):
        chunk = data[i:i+12]
        hex_vals = ', '.join(f'0x{b:02x}' for b in chunk)
        f.write(f'  {hex_vals},\n')
    f.write(f'}};\n\n')
    f.write(f'#endif // FALLNET_MODEL_H\n')

print(f"✅ C header saved: {header_path}")
print(f"   Array: fallnet_model[{len(data)}] ({len(data)/1024:.1f} KB)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
names = list(valid_results.keys())
sizes = [valid_results[n]['size_kb'] for n in names]
accs = [valid_results[n]['accuracy'] * 100 for n in names]
fi_recalls = [valid_results[n]['fall_init_recall'] * 100 for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336'][:len(names)]

ax1 = axes[0]
bars = ax1.bar(names, sizes, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Model Size (KB)', fontweight='bold')
ax1.set_title('Model Size by Quantization', fontweight='bold')
ax1.axhline(y=600, color='red', linestyle='--', alpha=0.7, label='Flash budget')
ax1.legend()
ax1_twin = ax1.twinx()
ax1_twin.plot(names, accs, 'ko-', markersize=8, linewidth=2, label='Accuracy %')
ax1_twin.set_ylabel('Accuracy (%)', fontweight='bold')
ax1_twin.legend(loc='center right')

ax2 = axes[1]
ax2.bar(names, fi_recalls, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Fall_Initiation Recall (%)', fontweight='bold')
ax2.set_title('Safety Metric by Quantization', fontweight='bold')
ax2.axhline(y=95, color='red', linestyle='--', alpha=0.7, label='95% threshold')
ax2.axhline(y=99, color='green', linestyle='--', alpha=0.5, label='99% target')
ax2.set_ylim(min(fi_recalls) - 3, 101)
ax2.legend()
for i, r in enumerate(fi_recalls):
    ax2.text(i, r + 0.3, f'{r:.1f}%', ha='center', fontweight='bold')

plt.suptitle('FallNet CNN→LMU (StaticLMU) — Quantization Tradeoffs', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / 'quantization_comparison_staticlmu.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Quantization comparison chart saved")


# %% [markdown]
## 14. Final Summary

# %%
stats = fallnet.get_model_stats()

print(f"\n{'='*80}")
print("TRAINING + QUANTIZATION COMPLETE — FINAL SUMMARY")
print(f"{'='*80}")

summary = f"""
✅ FallNet {MODEL_VARIANT.upper()} with StaticLMU — 5-fold CV + Quantization

Architecture: CNN→StaticLMU Hybrid (TFLite-compatible, static unrolling)
  - StaticLMU replaces keras_lmu.LMU (eliminates TensorListReserve ops)
  - Identical LMU math (Legendre memory), different execution strategy

Model: {stats['total']:,} params

Average Performance (5-fold CV):
  - Accuracy:  {mean_results['val_accuracy']:.4f} ± {std_results['val_accuracy']:.4f}
  - Precision: {mean_results['val_precision']:.4f} ± {std_results['val_precision']:.4f}
  - Recall:    {mean_results['val_recall']:.4f} ± {std_results['val_recall']:.4f}
  - F1-Score:  {mean_results['val_f1']:.4f} ± {std_results['val_f1']:.4f}

Fall_Initiation (Keras): Recall={keras_fi_recall:.4f}, F1={keras_fi_f1:.4f}

Quantization Results:
"""

for name, r in valid_results.items():
    summary += f"  {name:<16s}: {r['size_kb']:>7.1f} KB | Acc: {r['accuracy']:.4f} | FI Recall: {r['fall_init_recall']:.4f}\n"

summary += f"""
Deployment Model: {best_quant}
  Size: {valid_results[best_quant]['size_kb']:.1f} KB
  Accuracy: {valid_results[best_quant]['accuracy']:.4f}
  FI Recall: {valid_results[best_quant]['fall_init_recall']:.4f}

Target: Arduino Nano 33 BLE Sense Rev2 (nRF52840, 1MB flash, 256KB RAM)
Files: fallnet_hybrid_staticlmu_{best_quant}.tflite, fallnet_model.h
"""

print(summary)
with open(output_dir / f'training_summary_{MODEL_VARIANT}_staticlmu.txt', 'w') as f:
    f.write(summary)
print(f"✅ Summary saved")
print(f"\n🎯 Next: Flash fallnet_model.h to Nano BLE Sense, measure power with PPK2")

In [ ]:
from pathlib import Path
models_dir = Path('~/repos/summerschool2023/projects/fall-detection/fall_detection_data/models').expanduser()
print(list(models_dir.glob('*.keras')))

In [ ]:
# --- Weight Analysis: LMUFFT-SNN ---

# Load best fold (fold 1 was highest at 84.05%)
best_fold = 1
model_inspect = FallNet_LMUFFT_SNN(
    num_steps=NUM_STEPS, lmu_hidden=LMU_HIDDEN, lmu_order=LMU_ORDER
).to(device)
model_inspect.load_state_dict(torch.load(
    lmu_snn_dir / f'lmufft_snn_fold_{best_fold}.pth', weights_only=True
))
model_inspect.eval()

print("=" * 80)
print("LEARNED BETA VALUES (LIF decay rates, init=0.95)")
print("=" * 80)
print(f"  lif1 beta: {model_inspect.lif1.beta.item():.4f}  (128-neuron layer)")
print(f"  lif2 beta: {model_inspect.lif2.beta.item():.4f}  (64-neuron layer)")
print(f"  lif3 beta: {model_inspect.lif3.beta.item():.4f}  (output layer)")

print("\n" + "=" * 80)
print("LMU ENCODER WEIGHTS (per IMU channel)")
print("=" * 80)
channel_names = ['Acc-X', 'Acc-Y', 'Acc-Z', 'Gyr-X', 'Gyr-Y', 'Gyr-Z']
print(f"\n  Encoder weights (scalar drive, W_x):")
for c, name in enumerate(channel_names):
    w = model_inspect.lmu_encoder.encoders[c].weight.item()
    b = model_inspect.lmu_encoder.encoders[c].bias.item()
    print(f"    {name}: weight={w:+.4f}  bias={b:+.4f}")

print(f"\n  Read-out weight norms (which Legendre coefficients matter):")
for c, name in enumerate(channel_names):
    W = model_inspect.lmu_encoder.readouts[c].weight  # [hidden, order]
    col_norms = W.norm(dim=0)  # norm per Legendre coefficient
    top_coeff = col_norms.argmax().item()
    print(f"    {name}: per-order norms = {col_norms.detach().cpu().numpy().round(3)}")
    print(f"           most important Legendre order: {top_coeff} "
          f"({'low-freq trend' if top_coeff < 3 else 'mid-freq' if top_coeff < 6 else 'high-freq'})")

print("\n" + "=" * 80)
print("SPIKE RATE ANALYSIS (sparsity = energy efficiency proxy)")
print("=" * 80)

# Run a batch through and measure spike rates
sample_data, sample_targets = next(iter(DataLoader(dataset, batch_size=256, shuffle=False)))
sample_data = sample_data.to(device)

with torch.no_grad():
    # Temporarily hook into forward to capture per-layer spikes
    spk_out, mem_out = model_inspect(sample_data)

# spk_out: [num_steps, batch, classes] — output layer
spk_rate_out = spk_out.mean().item()

print(f"\n  Output layer mean spike rate: {spk_rate_out:.3f}")
print(f"  Sparsity: {(1-spk_rate_out)*100:.1f}% of output neurons silent per step")
print(f"\n  (For per-layer rates, add hooks to lif1/lif2 — see below)")

# Per-layer hook version
rates = {}
def make_hook(name):
    def hook(module, input, output):
        spk = output[0] if isinstance(output, tuple) else output
        rates[name] = spk.mean().item()
    return hook

h1 = model_inspect.lif1.register_forward_hook(make_hook('lif1'))
h2 = model_inspect.lif2.register_forward_hook(make_hook('lif2'))
h3 = model_inspect.lif3.register_forward_hook(make_hook('lif3'))

with torch.no_grad():
    _ = model_inspect(sample_data)

h1.remove(); h2.remove(); h3.remove()

print(f"\n  Per-layer spike rates:")
for name, rate in rates.items():
    print(f"    {name}: {rate:.3f}  ({(1-rate)*100:.1f}% sparse)")

print(f"\n  MAC savings estimate vs dense ANN:")
total_ops = 192*128 + 128*64 + 64*6
for name, rate in rates.items():
    print(f"    {name}: effective ops = {rate:.3f}× dense")

In [ ]:
# %% [markdown]
# ## FallNet Phase 2: LMU-SNN Hybrid
#
# Architecture:
#   IMU [batch, 6, 200]
#     → LMUEncoder (one LMUCell per IMU channel, processes 200 timesteps)
#     → [batch, lmu_hidden × 6]  compact Legendre temporal encoding
#     → Spiking FC classifier (3 × Linear → BN → LIF, integrated over num_steps)
#     → Spike accumulation → [batch, 6] class logits
#
# Theory: Fourier → Wavelets → LMU (Padé/Legendre basis) → SNN (LIF integration)

# %%
# --- Cell 1: Imports ---

import torch
import torch.nn as nn
import torch.nn.functional as F
import snntorch as snn
from snntorch import surrogate
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import numpy as np
from scipy.signal import cont2discrete

print("✅ Imports OK")

# %%
# --- Cell 2: LMU Cell ---

class LMUCell(nn.Module):
    """
    Single-step Legendre Memory Unit.

    Per timestep:
        e_t  = tanh(W_x @ x_t + W_h @ h_{t-1} + W_m @ m_{t-1})   # encoder
        m_t  = A_bar @ m_{t-1} + B_bar @ e_t                       # memory update
        h_t  = tanh(H_x @ x_t + H_m @ m_t)                        # hidden state

    A_bar, B_bar are analytically derived (Voelker et al. 2019) via
    zero-order-hold discretisation of the continuous Legendre delay ODE.
    They are frozen by default — the network learns only the coupling weights.
    """

    def __init__(self, input_size, hidden_size, order, theta, learn_ab=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.order = order

        # --- Derive A_bar, B_bar analytically ---
        Q = np.arange(order, dtype=np.float64)
        R = (2 * Q + 1)[:, None]
        j, i = np.meshgrid(Q, Q)
        A_cont = np.where(i < j, -1, (-1.0) ** (i - j + 1)) * R / theta
        B_cont = (-1.0) ** Q[:, None] * R / theta

        C = np.zeros((1, order))
        D = np.zeros((1,))
        A_bar, B_bar, _, _, _ = cont2discrete(
            (A_cont, B_cont, C, D), dt=1.0, method='zoh'
        )

        if learn_ab:
            self.A_bar = nn.Parameter(torch.FloatTensor(A_bar))
            self.B_bar = nn.Parameter(torch.FloatTensor(B_bar))
        else:
            self.register_buffer('A_bar', torch.FloatTensor(A_bar))
            self.register_buffer('B_bar', torch.FloatTensor(B_bar))

        # Encoder weights
        self.e_x = nn.Linear(input_size,  1, bias=False)
        self.e_h = nn.Linear(hidden_size, 1, bias=False)
        self.e_m = nn.Linear(order,       1, bias=False)

        # Hidden weights
        self.h_x = nn.Linear(input_size, hidden_size, bias=False)
        self.h_m = nn.Linear(order,      hidden_size, bias=False)

    def forward(self, x, state):
        h, m = state
        u     = torch.tanh(self.e_x(x) + self.e_h(h) + self.e_m(m))   # [B, 1]
        m_new = F.linear(m, self.A_bar) + F.linear(u, self.B_bar.T)    # [B, order]
        h_new = torch.tanh(self.h_x(x) + self.h_m(m_new))              # [B, hidden]
        return h_new, (h_new, m_new)

    def init_state(self, batch_size, device):
        return (
            torch.zeros(batch_size, self.hidden_size, device=device),
            torch.zeros(batch_size, self.order,       device=device),
        )


print("✅ LMUCell defined")

# %%
# --- Cell 3: LMU Encoder ---

class LMUEncoder(nn.Module):
    """
    Runs one LMUCell per IMU channel over the full 200-timestep window.
    Each channel independently learns its Legendre memory representation.
    Outputs are concatenated: [batch, hidden_size * in_channels].
    """

    def __init__(self, in_channels=6, hidden_size=32, order=8,
                 theta=200, learn_ab=False):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_size = hidden_size
        self.output_size = hidden_size * in_channels

        self.lmu_cells = nn.ModuleList([
            LMUCell(1, hidden_size, order, theta, learn_ab)
            for _ in range(in_channels)
        ])

    def forward(self, x):
        # x: [batch, in_channels, seq_len]
        batch_size = x.size(0)
        device     = x.device

        states = [cell.init_state(batch_size, device) for cell in self.lmu_cells]

        for t in range(x.size(2)):                          # step over 200 timesteps
            for c, cell in enumerate(self.lmu_cells):
                _, states[c] = cell(x[:, c:c+1, t], states[c])

        h_finals = [states[c][0] for c in range(self.in_channels)]
        return torch.cat(h_finals, dim=-1)                  # [batch, hidden * channels]


print("✅ LMUEncoder defined")

# %%
# --- Cell 4: FallNet LMU-SNN Model ---

class FallNet_LMU_SNN(nn.Module):
    """
    Stage 1 (LMU):  Encodes 200-timestep IMU window into Legendre state
    Stage 2 (SNN):  Integrates that state over num_steps via spiking LIF layers
    """

    def __init__(
        self,
        num_classes = 6,
        num_steps   = 25,
        lmu_hidden  = 32,    # hidden size per channel → 192 total (32×6)
        lmu_order   = 8,     # Legendre polynomial order
        lmu_theta   = 200,   # match your sequence length
        beta        = 0.95,
        threshold   = 1.0,
        learn_ab    = False,
    ):
        super().__init__()
        self.num_steps = num_steps
        spike_grad = surrogate.fast_sigmoid(slope=25)

        # Stage 1: LMU encoder
        self.lmu_encoder = LMUEncoder(
            in_channels=6,
            hidden_size=lmu_hidden,
            order=lmu_order,
            theta=lmu_theta,
            learn_ab=learn_ab,
        )
        lmu_out = lmu_hidden * 6  # 192 with defaults

        # Stage 2: Spiking classifier
        self.fc1  = nn.Linear(lmu_out, 128)
        self.bn1  = nn.BatchNorm1d(128)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad,
                               threshold=threshold, learn_beta=True)

        self.fc2  = nn.Linear(128, 64)
        self.bn2  = nn.BatchNorm1d(64)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad,
                               threshold=threshold, learn_beta=True)

        self.fc3  = nn.Linear(64, num_classes)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad,
                               threshold=threshold, learn_beta=True)

    def forward(self, x):
        # LMU encoding runs once (not per SNN step)
        lmu_out = self.lmu_encoder(x)           # [batch, 192]

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()

        spk_rec, mem_rec = [], []

        for _ in range(self.num_steps):
            spk1, mem1 = self.lif1(self.bn1(self.fc1(lmu_out)), mem1)
            spk2, mem2 = self.lif2(self.bn2(self.fc2(spk1)),    mem2)
            spk3, mem3 = self.lif3(self.fc3(spk2),              mem3)
            spk_rec.append(spk3)
            mem_rec.append(mem3)

        return torch.stack(spk_rec), torch.stack(mem_rec)


# Sanity check
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_model     = FallNet_LMU_SNN().to(device)
_x         = torch.randn(4, 6, 200).to(device)
with torch.no_grad():
    _spk, _mem = _model(_x)

total_p = sum(p.numel() for p in _model.parameters())
lmu_p   = sum(p.numel() for p in _model.lmu_encoder.parameters())
print(f"✅ FallNet_LMU_SNN defined")
print(f"   Input:  {list(_x.shape)}")
print(f"   Output: {list(_spk.shape)}  [steps, batch, classes]")
print(f"   Params: {total_p:,} total  ({lmu_p:,} LMU + {total_p-lmu_p:,} SNN)")
del _model, _x, _spk, _mem

# %%
# --- Cell 5: Dataset + Class Weights ---

class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Assumes X_data_torch, y_labels_torch already defined from your preprocessing cell
cw = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(6),
    y=y_labels_torch.numpy()
)
cw = np.clip(cw, None, 3.0)
cw_tensor = torch.FloatTensor(cw).to(device)

print("Class weights (capped at 3×):")
for i, w in enumerate(cw):
    print(f"  {reverse_label_map[i]:<30s}: {w:.3f}×")

dataset = FallDataset(X_data_torch, y_labels_torch)
print(f"\n✅ Dataset ready: {len(dataset):,} samples")

# %%
# --- Cell 6: Training Config ---

BATCH_SIZE = 32     # smaller than pure-SNN due to LMU sequential cost
EPOCHS     = 30
LR         = 5e-4
NUM_STEPS  = 25
LMU_HIDDEN = 32     # 32 × 6 channels = 192 features into SNN
LMU_ORDER  = 8      # Legendre polynomial order; try 4 or 16 to tune
N_FOLDS    = 5

lmu_snn_dir = models_dir / 'lmu_snn'
lmu_snn_dir.mkdir(exist_ok=True)

print("=" * 80)
print("FallNet LMU-SNN — 5-Fold Cross-Validation")
print("=" * 80)
print(f"Device:     {device}")
print(f"Batch:      {BATCH_SIZE} | Epochs: {EPOCHS} | SNN steps: {NUM_STEPS}")
print(f"LMU hidden: {LMU_HIDDEN}/channel | LMU order: {LMU_ORDER}")
print(f"LMU output: {LMU_HIDDEN * 6} features → SNN")

# %%
# --- Cell 7: 5-Fold Training Loop ---

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = {
    'val_acc':          [],
    'fall_init_recall': [],
    'predictions':      [],
    'targets':          [],
}

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_data_torch, y_labels_torch), 1
):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{N_FOLDS} — train: {len(train_idx):,}  val: {len(val_idx):,}")
    print(f"{'='*80}")

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True,
    )

    model     = FallNet_LMU_SNN(
        num_steps=NUM_STEPS, lmu_hidden=LMU_HIDDEN, lmu_order=LMU_ORDER
    ).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        # ---- Train ----
        model.train()
        epoch_loss = 0.0

        for data, targets in train_loader:
            data, targets = data.to(device), targets.to(device)
            spk_out, mem_out = model(data)

            # Combined loss: membrane average + 0.5 × spike count
            loss = (criterion(mem_out.mean(dim=0), targets)
                    + 0.5 * criterion(spk_out.sum(dim=0), targets))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()

        # ---- Validate ----
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for data, targets in val_loader:
                data, targets = data.to(device), targets.to(device)
                predicted = model(data)[0].sum(dim=0).argmax(dim=1)
                total    += targets.size(0)
                correct  += (predicted == targets).sum().item()

        val_acc = correct / total
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(),
                       lmu_snn_dir / f'lmu_snn_fold_{fold}.pth')

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:2d}/{EPOCHS} | "
                  f"Loss: {epoch_loss/len(train_loader):.4f} | "
                  f"Val: {val_acc:.4f} | Best: {best_acc:.4f}")

    # ---- Final fold eval ----
    model.load_state_dict(torch.load(
        lmu_snn_dir / f'lmu_snn_fold_{fold}.pth', weights_only=True
    ))
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for data, targets in val_loader:
            predicted = model(data.to(device))[0].sum(dim=0).argmax(dim=1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.numpy())

    all_preds   = np.array(all_preds)
    all_targets = np.array(all_targets)

    fi_idx    = label_map['Fall_Initiation']
    fi_mask   = all_targets == fi_idx
    fi_recall = (all_preds[fi_mask] == fi_idx).mean() if fi_mask.any() else 0.0

    fold_results['val_acc'].append(best_acc)
    fold_results['fall_init_recall'].append(fi_recall)
    fold_results['predictions'].append(all_preds)
    fold_results['targets'].append(all_targets)

    print(f"\n  Fold {fold} → Acc: {best_acc*100:.2f}%  "
          f"Fall_Init Recall: {fi_recall*100:.2f}%")

# %%
# --- Cell 8: Results ---

mean_acc    = np.mean(fold_results['val_acc'])
std_acc     = np.std(fold_results['val_acc'])
mean_recall = np.mean(fold_results['fall_init_recall'])
std_recall  = np.std(fold_results['fall_init_recall'])

print("=" * 80)
print("5-FOLD RESULTS — FallNet LMU-SNN")
print("=" * 80)
for i, (a, r) in enumerate(zip(
    fold_results['val_acc'], fold_results['fall_init_recall']
), 1):
    print(f"  Fold {i}: Acc {a*100:.2f}%  Fall_Init Recall {r*100:.2f}%")

print(f"\nMean Accuracy:        {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Mean Fall_Init Recall:{mean_recall*100:.2f}% ± {std_recall*100:.2f}%")

print("\n" + "=" * 80)
print("COMPARISON")
print("=" * 80)
print(f"{'Model':<25} {'Accuracy':<20} {'Fall_Init Recall'}")
print("-" * 65)
print(f"{'CNN (FP32)':<25} {'94.71%':<20} {'97.82%'}")
print(f"{'SNN (trained)':<25} {'88.83%':<20} {'see cv_results.json'}")
print(f"{'LMU-SNN':<25} "
      f"{mean_acc*100:.2f}% ± {std_acc*100:.2f}%    "
      f"{mean_recall*100:.2f}% ± {std_recall*100:.2f}%")

print("\n" + "=" * 80)
print("AGGREGATE CLASSIFICATION REPORT")
print("=" * 80)
all_p = np.concatenate(fold_results['predictions'])
all_t = np.concatenate(fold_results['targets'])
names = [reverse_label_map[i] for i in range(6)]
print(classification_report(all_t, all_p, target_names=names, digits=4))
